# Model-agnostic check — the rejection rule on a SECOND detector

The paper claims the risk-controlled rejection rule works on ANY frozen
detector, not just GR-ConvNet. This notebook backs that claim by running the
SAME pipeline on TWO more detectors -- **GG-CNN** (plain, tiny) and **U-Net**
(skip connections) -- and comparing the certified operating points across all
three. The three span distinct design philosophies: residual (GR-ConvNet), plain
(GG-CNN), and skip-connected (U-Net), so agreement is a real across-architecture
result, not three near-duplicates.

If the rejection guarantee holds and the risk-coverage story is similar on both
detectors, the method is demonstrably model-agnostic. That is a stronger,
lower-risk novelty than adding another uncertainty score.

**Cost.** GG-CNN is ~10x smaller than GR-ConvNet, so it trains in minutes. You
need the Cornell data (same as the main notebook) and, ideally, the GR-ConvNet
`scores.npz` already on Drive so the two can be compared side by side.

In [ ]:
!nvidia-smi || echo "no GPU — Runtime > Change runtime type > GPU"

## 1. Dependencies + Drive

In [ ]:
import importlib
for mod, pkg in [('numpy','numpy'),('scipy','scipy'),('PIL','pillow'),
                 ('yaml','pyyaml'),('matplotlib','matplotlib'),
                 ('tifffile','tifffile'),('imageio','imageio')]:
    try: importlib.import_module(mod)
    except Exception: 
        !pip install -q {pkg}
import numpy as np, torch
print('torch', torch.__version__, 'cuda', torch.cuda.is_available())

### Choose the dataset

The model-agnostic study runs on EITHER dataset. Set `DATASET` and the rest of
the notebook (configs, paths, the GR-ConvNet reference cache) follows. Run it
once for 'cornell' and once for 'jacquard' to show the guarantee holds across
detectors AND datasets.

In [ ]:
DATASET = 'cornell'      # <-- 'cornell' or 'jacquard'

from google.colab import drive
drive.mount('/content/drive')
import os

# per-dataset Drive folder and local data dir; matches the main notebooks
DRIVE = {'cornell': '/content/drive/MyDrive/PhD/fisat',
         'jacquard':'/content/drive/MyDrive/PhD/fisat_jacquard'}[DATASET]
DATA  = {'cornell': '/content/data/cornell',
         'jacquard':'/content/data/jacquard'}[DATASET]
WORK  = f'/content/fisat_agnostic_{DATASET}'
for p in (WORK, DATA): os.makedirs(p, exist_ok=True)
os.chdir(WORK)
print('dataset:', DATASET, '| cwd', os.getcwd(), '| drive', DRIVE)

## 2. Write the code (GG-CNN included)

In [ ]:
%%writefile data_cornell.py
"""
src/data/cornell.py
===================
Drop-in Cornell dataset for the grasp-UQ benchmark. Matches the constructor
signature and batch format of src.data.jacquard.JacquardV2 so that switching the
FISAT experiments from Jacquard V2 to Cornell is a one-line import change in the
notebook (and the same `collate`).

Why Cornell for the FISAT paper
-------------------------------
The backbone checkpoints were lost, so the detector must be retrained from
scratch. Cornell is ~1k real RGB-D images (240 objects) vs Jacquard's ~54k
simulated scenes, so it trains in hours rather than days — decisive under the
10/08 deadline — and its real-object images make the qualitative accept/reject
figures more convincing. The one cost is calibration sample size; see the
split note at the bottom.

Expected on-disk layout (standard Cornell release)
--------------------------------------------------
    <root>/
        01/  pcd0100r.png  pcd0100d.tiff  pcd0100cpos.txt  pcd0100cneg.txt ...
        02/  ...
        ...
        10/

    *r.png    : RGB image (640x480)
    *d.tiff   : depth image, OR a pcdXXXX.txt point cloud to be projected
    *cpos.txt : positive grasp rectangles, 4 corner points per grasp (2 lines
                each pair -> 4 lines per rectangle), pixel coords "x y"
    *cneg.txt : negative rectangles (unused for success labels here)

If your copy ships depth as .txt point clouds rather than .tiff, set
`depth_from_pcd=True`; the loader will project them.

Grasp representation
--------------------
`gt_grasps` is returned in the SAME structure the repo already consumes for
Jacquard (an (N,4,2) float array of rectangle corners per sample, or whatever
`gt_array_to_grasps` expects — adapt `_load_grasp_rects` if the repo uses a
different convention). Keeping this identical is what makes the loader a true
drop-in.
"""

from __future__ import annotations

import glob
import os

import numpy as np

try:
    import torch
    from torch.utils.data import Dataset
    _HAS_TORCH = True
except Exception:                                     # pragma: no cover
    _HAS_TORCH = False
    Dataset = object

# ---------------------------------------------------------------------------
# Backend-agnostic image IO.
#
# The first version of this file did `try: from skimage.io import imread /
# except: imread = None`, which turned a clear ImportError at module load into
# a baffling "TypeError: 'NoneType' object is not callable" thrown from inside
# a data loader many frames away. Swallowing an import error and continuing
# with a None is strictly worse than failing loudly.
#
# Instead: try every backend that might be present, and if all of them fail,
# raise ONE error that names each backend and why it failed.
# ---------------------------------------------------------------------------
def _imread(path: str) -> np.ndarray:
    errors = []
    if path.lower().endswith((".tif", ".tiff")):
        try:
            import tifffile
            return np.asarray(tifffile.imread(path))
        except Exception as e:
            errors.append(f"tifffile: {e}")
    try:
        from PIL import Image
        return np.asarray(Image.open(path))
    except Exception as e:
        errors.append(f"PIL: {e}")
    try:
        from skimage.io import imread as _sk_imread
        return np.asarray(_sk_imread(path))
    except Exception as e:
        errors.append(f"skimage: {e}")
    try:
        import imageio.v2 as iio
        return np.asarray(iio.imread(path))
    except Exception as e:
        errors.append(f"imageio: {e}")
    raise RuntimeError(
        f"No image backend could read {path}.\nTried:\n  " + "\n  ".join(errors)
        + "\n\nInstall one of: pillow, tifffile, scikit-image, imageio.")


def _resize(arr: np.ndarray, out_size: int) -> np.ndarray:
    """Bilinear resize to (out_size, out_size), preserving dtype range.

    Uses scipy.ndimage.zoom rather than skimage: it handles float arrays and
    channel dimensions natively, and scipy is already required elsewhere in the
    pipeline, so this removes a dependency instead of adding one.
    """
    h, w = arr.shape[:2]
    if (h, w) == (out_size, out_size):
        return arr.astype(np.float32)
    from scipy.ndimage import zoom
    factors = [out_size / h, out_size / w]
    if arr.ndim == 3:
        factors.append(1.0)
    return zoom(arr.astype(np.float32), factors, order=1)


def available_backends() -> dict:
    """Report which image backends import cleanly. Call this when IO misbehaves."""
    out = {}
    for name, mod in [("PIL", "PIL.Image"), ("tifffile", "tifffile"),
                      ("skimage", "skimage.io"), ("imageio", "imageio"),
                      ("scipy", "scipy.ndimage")]:
        try:
            __import__(mod)
            out[name] = "ok"
        except Exception as e:
            out[name] = f"FAILED: {type(e).__name__}: {e}"
    return out


# --------------------------------------------------------------------------- #
# Cornell rectangle parsing
# --------------------------------------------------------------------------- #
def _read_cornell_rects(path: str) -> list[np.ndarray]:
    """Parse a Cornell cpos/cneg file into a list of (4,2) corner arrays.

    The file lists 4 lines per rectangle: 'x y' each. Lines containing NaN
    (annotation gaps) invalidate their whole 4-line block, matching the standard
    Cornell handling.
    """
    rects, buf = [], []
    if not os.path.exists(path):
        return rects
    with open(path) as f:
        for line in f:
            parts = line.split()
            if len(parts) != 2:
                continue
            try:
                x, y = float(parts[0]), float(parts[1])
            except ValueError:
                buf = []
                continue
            if np.isnan(x) or np.isnan(y):
                buf = []
                continue
            buf.append((x, y))
            if len(buf) == 4:
                rects.append(np.array(buf, dtype=np.float32))  # (4,2) in (x,y)
                buf = []
    return rects


def _rect_center_angle_wh(rect: np.ndarray):
    """(4,2) corners -> (cx, cy, theta, w, h). theta is the gripper-plate angle."""
    cx, cy = rect[:, 0].mean(), rect[:, 1].mean()
    # edge from corner 0->1 is one gripper plate direction
    dx, dy = rect[1, 0] - rect[0, 0], rect[1, 1] - rect[0, 1]
    theta = np.arctan2(dy, dx)
    w = np.hypot(dx, dy)
    dx2, dy2 = rect[2, 0] - rect[1, 0], rect[2, 1] - rect[1, 1]
    h = np.hypot(dx2, dy2)
    return float(cx), float(cy), float(theta), float(w), float(h)


# --------------------------------------------------------------------------- #
# Dataset
# --------------------------------------------------------------------------- #
class Cornell(Dataset):
    """Cornell grasp dataset with the JacquardV2 constructor signature.

    Signature mirrors:
        JacquardV2(root, split, out_size, use_depth, mode,
                   val_frac, novel_frac, test_frac, seed)

    Splits are OBJECT-DISJOINT. Cornell has ~240 distinct objects; `object_id`
    is parsed from the standard file grouping so no image of an object leaks
    across splits. This is what lets the paper claim the risk guarantee holds on
    unseen objects.
    """

    def __init__(self, root, split="train", out_size=224, use_depth=True,
                 mode="real", val_frac=0.15, novel_frac=0.0, test_frac=0.2,
                 seed=0, depth_from_pcd=False, split_mode='object'):
        self.root = root
        self.split = split
        self.out_size = out_size
        self.use_depth = use_depth
        self.depth_from_pcd = depth_from_pcd

        # NOTE: the release ships a `backgrounds/` folder whose files are named
        # pcdb0001r.png etc. Those MATCH a naive pcd*r.png glob and would be
        # silently loaded as training samples with empty grasp sets. We exclude
        # that folder AND require a matching cpos.txt, so a background image
        # cannot slip in through either path.
        all_rgb = sorted(glob.glob(os.path.join(root, "**", "pcd*r.png"),
                                   recursive=True))
        rgb_files = []
        for pth in all_rgb:
            parts = pth.replace("\\", "/").split("/")
            if any(seg.lower() == "backgrounds" for seg in parts):
                continue
            if os.path.basename(pth).startswith("pcdb"):
                continue
            if not os.path.exists(pth.replace("r.png", "cpos.txt")):
                continue
            rgb_files.append(pth)

        if not rgb_files:
            raise FileNotFoundError(
                f"No Cornell RGB files (pcd*r.png with a matching cpos.txt) "
                f"under {root}. Check the dataset was unzipped with its "
                f"01..10 subfolders.")

        # ---- grouping for the object-disjoint split ------------------------ #
        # Cornell carries NO explicit object id. The raw numbering (pcd0100..
        # pcd1034, ~885 images over ~240 objects) has each object as a short
        # contiguous run, but the run boundaries are not recorded anywhere, so
        # any per-object id would be a guess.
        #
        # We therefore group by the release's own numbered FOLDER (01..10).
        # This is coarse -- 10 groups, not 240 -- but it is genuinely disjoint:
        # folders are contiguous number ranges, so no object spans two folders
        # except possibly at a boundary. Claiming per-object ids we cannot
        # verify would be worse than being coarse and honest.
        #
        # split_mode='image' is offered for the calibration/evaluation stage,
        # where exchangeability (not object novelty) is what the LTT guarantee
        # actually requires. See SPLIT_NOTE at the bottom.
        def folder_id(pth):
            parts = pth.replace("\\", "/").split("/")
            for seg in reversed(parts[:-1]):
                if seg.isdigit():
                    return int(seg)
            return 0

        self.split_mode = split_mode
        if split_mode == "object":
            group = np.array([folder_id(p) for p in rgb_files])
        elif split_mode == "image":
            group = np.arange(len(rgb_files))
        else:
            raise ValueError("split_mode must be 'object' or 'image'")

        uniq = np.unique(group)
        rng = np.random.default_rng(seed)
        uniq = rng.permutation(uniq)
        n = len(uniq)
        n_test = max(1, int(round(test_frac * n))) if test_frac > 0 else 0
        n_val = max(1, int(round(val_frac * n))) if val_frac > 0 else 0
        n_novel = int(round(novel_frac * n))

        test_g = set(uniq[:n_test])
        val_g = set(uniq[n_test:n_test + n_val])
        novel_g = set(uniq[n_test + n_val:n_test + n_val + n_novel])
        train_g = set(uniq[n_test + n_val + n_novel:])

        sel = {"train": train_g, "val": val_g,
               "test": test_g, "novel": novel_g}[split]
        keep = np.array([g in sel for g in group])
        self.files = [p for p, k in zip(rgb_files, keep) if k]
        self.obj_ids = [int(g) for g, k in zip(group, keep) if k]
        self.folder_ids = [folder_id(p) for p in self.files]
        if not self.files:
            raise ValueError(
                f"Cornell split '{split}' is empty (groups={n}, "
                f"test={n_test} val={n_val} novel={n_novel}). "
                f"With split_mode='object' there are only ~10 groups, so small "
                f"fracs can round to zero -- raise the frac or use "
                f"split_mode='image'.")

    def __len__(self):
        return len(self.files)

    # ---- loading helpers ------------------------------------------------- #
    def _load_rgb(self, rgb_path):
        img = _imread(rgb_path).astype(np.float32)
        if img.ndim == 2:
            img = np.stack([img] * 3, -1)
        img = img[..., :3]                       # drop alpha if present
        return img / 255.0

    def _load_depth(self, rgb_path):
        d_path = rgb_path.replace("r.png", "d.tiff")
        if os.path.exists(d_path):
            d = _imread(d_path).astype(np.float32)
            if d.ndim == 3:
                d = d[..., 0]
        else:
            d = np.zeros(_imread(rgb_path).shape[:2], np.float32)
        # normalise depth robustly
        m = np.isfinite(d)
        if m.any():
            lo, hi = np.percentile(d[m], [2, 98])
            d = np.clip((d - lo) / (hi - lo + 1e-9), 0, 1)
        return d.astype(np.float32)

    def _center_crop_resize(self, arr):
        """Cornell is 640x480; center-crop to square then resize to out_size."""
        h, w = arr.shape[:2]
        s = min(h, w)
        top, left = (h - s) // 2, (w - s) // 2
        arr = arr[top:top + s, left:left + s]
        scale = self.out_size / s
        out = _resize(arr, self.out_size)
        return out.astype(np.float32), (top, left, scale)

    def _load_grasp_rects(self, rgb_path, crop):
        top, left, scale = crop
        pos = _read_cornell_rects(rgb_path.replace("r.png", "cpos.txt"))
        out = []
        for r in pos:
            rr = r.copy()
            rr[:, 0] = (rr[:, 0] - left) * scale     # x
            rr[:, 1] = (rr[:, 1] - top) * scale      # y
            # keep rectangles that survive the crop
            if (rr[:, 0].min() >= 0 and rr[:, 1].min() >= 0 and
                    rr[:, 0].max() < self.out_size and rr[:, 1].max() < self.out_size):
                out.append(rr)
        return np.array(out, dtype=np.float32) if out else np.zeros((0, 4, 2), np.float32)

    def __getitem__(self, idx):
        rgb_path = self.files[idx]
        rgb = self._load_rgb(rgb_path)
        rgb_c, crop = self._center_crop_resize(rgb)

        chans = [rgb_c.transpose(2, 0, 1)]           # (3,H,W)
        if self.use_depth:
            d = self._load_depth(rgb_path)
            d_c, _ = self._center_crop_resize(d[..., None])
            chans.append(d_c.transpose(2, 0, 1))     # (1,H,W)
        image = np.concatenate(chans, axis=0).astype(np.float32)

        gt = self._load_grasp_rects(rgb_path, crop)
        return {
            "image": torch.from_numpy(image),
            "gt_grasps": gt,
            "obj_id": int(self.obj_ids[idx]),
            "idx": idx,
        }


# --------------------------------------------------------------------------- #
# collate — mirror the repo's Jacquard collate so downstream code is unchanged
# --------------------------------------------------------------------------- #
def collate(batch):
    images = torch.stack([b["image"] for b in batch], 0)
    return {
        "image": images,
        "gt_grasps": [b["gt_grasps"] for b in batch],
        "obj_id": [b["obj_id"] for b in batch],
        "idx": [b["idx"] for b in batch],
    }


SPLIT_NOTE = """
Split discipline on Cornell (state this in Sec. 5 of the paper):

1. GROUPING. Cornell carries no explicit object id. The numbering (pcd0100..
   pcd1034) puts each object in a short contiguous run, but the run boundaries
   are not recorded, so any per-object id would be invented. We group by the
   release's numbered folders (01..10) instead: coarse (10 groups, not ~240)
   but genuinely disjoint, since folders are contiguous number ranges. Say this
   plainly rather than claiming object ids the dataset does not provide.

2. CONSEQUENCE FOR THE THREE-WAY SPLIT. With only 10 folder groups, a test
   partition is ~2 folders, which cannot be further divided three ways by
   folder. So:

       backbone train / val / test   -> split_mode='object'  (folder-level)
       gate-train / calibration / eval  -> split_mode='image' WITHIN test

   This is not a compromise forced by convenience. The LTT guarantee needs
   EXCHANGEABILITY between calibration and evaluation, which an image-level
   random split satisfies exactly; object novelty is what the train/test
   boundary is for, and that boundary stays folder-level. The two splits carry
   two different claims and it is worth spelling that out:

       train/test folder-disjoint -> the detector generalises to unseen objects
       cal/eval image-exchangeable -> the risk bound is valid on the test pool

3. SAMPLE SIZE. ~885 usable images total. After a folder-level test partition
   (~20% -> ~180 images) split three ways, calibration holds roughly 60 images,
   and the accepted subset at high coverage is smaller still. That is well under
   the ~150-200 needed to certify alpha=0.05 at delta=0.10.

   Mitigations, in order of preference:
     (a) enlarge the test partition (test_frac=0.4) since no backbone training
         happens on it in the FISAT paper -- the detector is frozen, so a big
         test partition costs only backbone accuracy, not validity;
     (b) report alpha=0.10 as the primary budget, alpha=0.05 as secondary;
     (c) if alpha=0.05 cannot be certified, report that as a finding: at this
         calibration size the achievable risk budget is bounded from below.
   Do NOT silently shrink n_min to force a certificate -- that breaks the very
   guarantee the paper is selling.
"""


In [ ]:
%%writefile data_jacquard.py
"""
data_jacquard.py
================
Jacquard V2 loader with the SAME constructor signature as the Cornell loader, so
train.py / extract_scores.py use it unchanged: swap the import and the config.

Jacquard V2 layout (from the official release, github.com/lqh12345/Jacquard_V2):

    JacquardV2_Dataset_<k>/
        <object_hash>/
            <view>_<hash>_grasps.txt        grasp annotations, one per line
            <view>_<hash>_RGB.png           RGB image (1024x1024)
            <view>_<hash>_perfect_depth.tiff   simulated depth (used here)
            <view>_<hash>_stereo_depth.tiff    generated stereo depth
            <view>_<hash>_mask.png          object segmentation mask

Two structural facts drive the design:

1. THE OBJECT ID IS IN THE PATH. Unlike Cornell, every file name carries the
   object hash, so an object-disjoint split is exact, not approximated by folder
   number. Different VIEWS of the same object must never straddle the split, so
   grouping is by hash, and all four views of an object move together.

2. GRASPS ARE ONE PER LINE as `x;y;theta;opening_width;jaw_size` (semicolons,
   theta in DEGREES, coordinates in the ORIGINAL image pixels). We convert each
   to the (4,2) corner array the rest of the pipeline expects, in the resized
   224x224 frame. jaw_size is the plate length; opening_width is the jaw
   opening -- the same opening/plate distinction the metric relies on.

Jacquard is ~51k images. `max_images` and `max_objects` let you work on a
manageable slice while wiring things up; leave them None for the full run.
"""

from __future__ import annotations

import glob
import os

import numpy as np

try:
    import torch
    from torch.utils.data import Dataset
    _HAS_TORCH = True
except Exception:                                     # pragma: no cover
    _HAS_TORCH = False
    Dataset = object


# --------------------------------------------------------------------------- #
# Backend-agnostic image IO (shared approach with the Cornell loader)
# --------------------------------------------------------------------------- #
def _imread(path: str) -> np.ndarray:
    errors = []
    if path.lower().endswith((".tif", ".tiff")):
        try:
            import tifffile
            return np.asarray(tifffile.imread(path))
        except Exception as e:
            errors.append(f"tifffile: {e}")
    try:
        from PIL import Image
        return np.asarray(Image.open(path))
    except Exception as e:
        errors.append(f"PIL: {e}")
    try:
        import imageio.v2 as iio
        return np.asarray(iio.imread(path))
    except Exception as e:
        errors.append(f"imageio: {e}")
    raise RuntimeError(f"No image backend could read {path}\n  " +
                       "\n  ".join(errors))


def _resize(arr: np.ndarray, out_size: int) -> np.ndarray:
    h, w = arr.shape[:2]
    if (h, w) == (out_size, out_size):
        return arr.astype(np.float32)
    from scipy.ndimage import zoom
    factors = [out_size / h, out_size / w]
    if arr.ndim == 3:
        factors.append(1.0)
    return zoom(arr.astype(np.float32), factors, order=1)


def available_backends() -> dict:
    out = {}
    for name, mod in [("PIL", "PIL.Image"), ("tifffile", "tifffile"),
                      ("imageio", "imageio"), ("scipy", "scipy.ndimage")]:
        try:
            __import__(mod); out[name] = "ok"
        except Exception as e:
            out[name] = f"FAILED: {type(e).__name__}: {e}"
    return out


# --------------------------------------------------------------------------- #
# Grasp-annotation parsing
# --------------------------------------------------------------------------- #
def parse_grasp_line(line: str):
    """One Jacquard grasp line -> (cx, cy, theta_rad, opening, jaw).

    Format: `x;y;theta;opening_width;jaw_size`, theta in degrees. Returns None
    for blank or malformed lines so a stray line can't abort a whole scene.
    """
    parts = line.strip().split(";")
    if len(parts) < 5:
        return None
    try:
        x, y, theta_deg, opening, jaw = (float(p) for p in parts[:5])
    except ValueError:
        return None
    return x, y, np.deg2rad(theta_deg), opening, jaw


def grasp_to_corners(cx, cy, theta, opening, jaw) -> np.ndarray:
    """(cx, cy, theta, opening, jaw) -> (4,2) corner array in (x, y).

    The rectangle's long edge is the plate length (jaw), the short edge is the
    opening. Corner order matches params_to_corners so corners_to_params
    recovers the same theta.
    """
    c, s = np.cos(theta), np.sin(theta)
    hw, hh = jaw / 2.0, opening / 2.0          # half plate, half opening
    base = np.array([[-hw, -hh], [hw, -hh], [hw, hh], [-hw, hh]], dtype=float)
    R = np.array([[c, -s], [s, c]], dtype=float)
    return base @ R.T + np.array([cx, cy], dtype=float)


# --------------------------------------------------------------------------- #
# Dataset
# --------------------------------------------------------------------------- #
class Jacquard(Dataset):
    """Jacquard V2, drop-in for the Cornell loader's signature.

    Signature intentionally matches Cornell(root, split, out_size, use_depth,
    mode, val_frac, novel_frac, test_frac, seed, split_mode) so train.py needs
    only its import changed.
    """

    def __init__(self, root, split="train", out_size=224, use_depth=True,
                 mode="real", val_frac=0.1, novel_frac=0.0, test_frac=0.1,
                 seed=0, split_mode="object", depth_kind="perfect",
                 max_images=None, max_objects=None):
        self.root = root
        self.out_size = out_size
        self.use_depth = use_depth
        self.depth_kind = depth_kind          # 'perfect' or 'stereo'

        rgbs = self._scan(root)
        if not rgbs:
            raise FileNotFoundError(
                f"No *_RGB.png found under {root}. Expected Jacquard V2 layout "
                f"JacquardV2_Dataset_*/<hash>/<view>_<hash>_RGB.png")

        # group by object hash so all views of an object share a split
        obj_of = {p: self._hash(p) for p in rgbs}
        objects = sorted(set(obj_of.values()))

        rng = np.random.default_rng(seed)
        if max_objects:
            objects = list(rng.permutation(objects)[:max_objects])
            keep = set(objects)
            rgbs = [p for p in rgbs if obj_of[p] in keep]

        if split_mode == "object":
            sel = self._object_split(objects, rgbs, obj_of, split,
                                     val_frac, test_frac, rng)
        else:
            sel = self._image_split(rgbs, split, val_frac, test_frac, rng)

        if max_images:
            sel = list(sel)[:max_images]

        self.samples = sorted(sel)
        self.obj_ids = [self._hash_to_int(obj_of[p]) for p in self.samples]

    # ---- scanning ------------------------------------------------------- #
    @staticmethod
    def _scan(root):
        # tolerate both the sharded (JacquardV2_Dataset_*) and flat layouts
        hits = glob.glob(os.path.join(root, "**", "*_RGB.png"), recursive=True)
        return sorted(hits)

    @staticmethod
    def _hash(rgb_path):
        # <view>_<hash>_RGB.png -> hash. The hash is everything between the
        # first underscore and the _RGB suffix, since the hash itself has no
        # underscores in the Jacquard naming.
        base = os.path.basename(rgb_path)[:-len("_RGB.png")]
        parts = base.split("_", 1)
        return parts[1] if len(parts) == 2 else base

    _hash_cache: dict = {}

    def _hash_to_int(self, h):
        # stable small int id per object hash, for arrays
        if h not in self._hash_cache:
            self._hash_cache[h] = len(self._hash_cache)
        return self._hash_cache[h]

    # ---- splitting ------------------------------------------------------ #
    def _object_split(self, objects, rgbs, obj_of, split, val_frac,
                      test_frac, rng):
        perm = list(rng.permutation(objects))
        n = len(perm)
        n_test = int(test_frac * n)
        n_val = int(val_frac * n)
        test_o = set(perm[:n_test])
        val_o = set(perm[n_test:n_test + n_val])
        train_o = set(perm[n_test + n_val:])
        pick = {"train": train_o, "val": val_o, "test": test_o}[split]
        return [p for p in rgbs if obj_of[p] in pick]

    def _image_split(self, rgbs, split, val_frac, test_frac, rng):
        perm = list(rng.permutation(rgbs))
        n = len(perm)
        n_test = int(test_frac * n)
        n_val = int(val_frac * n)
        segs = {"test": perm[:n_test],
                "val": perm[n_test:n_test + n_val],
                "train": perm[n_test + n_val:]}
        return segs[split]

    # ---- loading -------------------------------------------------------- #
    def _load_image(self, rgb_path):
        rgb = _imread(rgb_path).astype(np.float32)
        if rgb.ndim == 2:
            rgb = np.stack([rgb] * 3, -1)
        rgb = rgb[..., :3] / 255.0
        orig_h, orig_w = rgb.shape[:2]
        rgb = _resize(rgb, self.out_size) / (rgb.max() if rgb.max() > 1 else 1)

        if self.use_depth:
            suffix = f"_{self.depth_kind}_depth.tiff"
            d_path = rgb_path.replace("_RGB.png", suffix)
            if os.path.exists(d_path):
                d = _imread(d_path).astype(np.float32)
                if d.ndim == 3:
                    d = d[..., 0]
                d = _resize(d, self.out_size)
                # normalise depth to [0,1] robustly
                lo, hi = np.percentile(d, 1), np.percentile(d, 99)
                d = np.clip((d - lo) / (hi - lo + 1e-6), 0, 1)
            else:
                d = np.zeros((self.out_size, self.out_size), np.float32)
            img = np.concatenate([rgb, d[..., None]], -1)
        else:
            img = rgb
        return img, (orig_h, orig_w)

    def _load_grasps(self, rgb_path, orig_hw):
        g_path = rgb_path.replace("_RGB.png", "_grasps.txt")
        orig_h, orig_w = orig_hw
        sx = self.out_size / orig_w
        sy = self.out_size / orig_h
        rects = []
        if os.path.exists(g_path):
            for line in open(g_path):
                parsed = parse_grasp_line(line)
                if parsed is None:
                    continue
                cx, cy, theta, opening, jaw = parsed
                # scale center and sizes into the resized frame
                corners = grasp_to_corners(cx * sx, cy * sy, theta,
                                           opening * sy, jaw * sx)
                rects.append(corners)
        if not rects:
            return np.zeros((0, 4, 2), np.float32)
        return np.asarray(rects, np.float32)

    # ---- Dataset API ---------------------------------------------------- #
    def __len__(self):
        return len(self.samples)

    def __getitem__(self, i):
        rgb_path = self.samples[i]
        img, orig_hw = self._load_image(rgb_path)
        grasps = self._load_grasps(rgb_path, orig_hw)
        img_t = (torch.from_numpy(img).permute(2, 0, 1).float()
                 if _HAS_TORCH else img.transpose(2, 0, 1))
        return {"image": img_t, "gt_grasps": grasps,
                "obj_id": self.obj_ids[i], "idx": i}


# --------------------------------------------------------------------------- #
# Self-test (no torch, no dataset needed)
# --------------------------------------------------------------------------- #
def _selftest():
    def ok(c, m):
        print(f"  {'PASS' if c else 'FAIL'}  {m}")
        if not c:
            raise SystemExit(1)

    print("[grasp line parsing]")
    p = parse_grasp_line("512.0;480.5;45.0;60.0;120.0")
    ok(p is not None, "well-formed line parses")
    cx, cy, th, opening, jaw = p
    ok(cx == 512.0 and cy == 480.5, "center read correctly")
    ok(abs(th - np.deg2rad(45.0)) < 1e-9, "theta converted deg->rad")
    ok(opening == 60.0 and jaw == 120.0, "opening and jaw read correctly")
    ok(parse_grasp_line("") is None, "blank line -> None")
    ok(parse_grasp_line("1;2;3") is None, "short line -> None")
    ok(parse_grasp_line("a;b;c;d;e") is None, "non-numeric -> None")

    print("[corner construction]")
    from math import isclose
    corners = grasp_to_corners(100, 100, 0.0, 40, 80)
    ok(corners.shape == (4, 2), "four corners")
    # plate (jaw=80) is the long edge, opening=40 the short edge
    w_edge = np.hypot(*(corners[1] - corners[0]))
    h_edge = np.hypot(*(corners[2] - corners[1]))
    ok(isclose(w_edge, 80, abs_tol=1e-6), "long edge == jaw (plate) length")
    ok(isclose(h_edge, 40, abs_tol=1e-6), "short edge == opening")
    ok(isclose(corners.mean(0)[0], 100) and isclose(corners.mean(0)[1], 100),
       "centroid at the grasp center")

    print("[theta round trip through the shared metric]")
    # the metric's corners_to_params must recover the angle we put in
    import importlib.util
    if importlib.util.find_spec is not None:
        try:
            from metrics_grasp import corners_to_params, angle_diff
            for deg in (0, 30, 75, 120, 179):
                c = grasp_to_corners(100, 100, np.deg2rad(deg), 40, 80)
                _, _, rec_th, _, _ = corners_to_params(c)
                ok(angle_diff(rec_th, np.deg2rad(deg)) < 1e-6,
                   f"theta={deg}deg recovered by corners_to_params")
        except ImportError:
            print("  (metrics_grasp not importable here; skipped)")

    print("[hash extraction]")
    j = Jacquard.__new__(Jacquard)
    h = Jacquard._hash("/x/JacquardV2_Dataset_0/abc123def/0_abc123def_RGB.png")
    ok(h == "abc123def", f"object hash parsed from path (got {h})")
    h2 = Jacquard._hash("/x/2_abc123def_RGB.png")
    ok(h2 == "abc123def", "hash parsed regardless of view index")

    print("\nALL JACQUARD LOADER TESTS PASSED")


if __name__ == "__main__":
    print("torch available:", _HAS_TORCH)
    print("backends:", available_backends())
    _selftest()


In [ ]:
%%writefile models_grconvnet.py
"""
models/grconvnet.py
===================
GR-ConvNet (Kumra et al., "Antipodal Robotic Grasping using Generative Residual
Convolutional Neural Network"): a fully-convolutional encoder-residual-decoder
emitting four dense maps at input resolution.

    pos   : grasp quality logits (sigmoid at post-processing time)
    cos   : cos(2*theta)
    sin   : sin(2*theta)
    width : gripper opening, rescaled by 150 at decode time

The angle is stored as (cos 2theta, sin 2theta) BECAUSE the gripper angle is
pi-periodic: doubling maps theta and theta+pi onto the same point, so the
network never has to learn a discontinuity that does not physically exist.
Decode with theta = atan2(sin, cos) / 2.

>>> The heads are 1x1 convolutions, and that is load-bearing. <<<
With kernel_size=2, padding=1 they emit H+1 (225 from a 224 input); with
kernel_size=2, padding=0 they emit H-1. Neither crashes -- it silently
misaligns every dense target by a pixel and quietly degrades the metric.
shape_trace() below verifies the whole chain arithmetically, with no torch
required, so this class of bug is caught before any GPU time is spent.
"""

from __future__ import annotations

# torch is imported optionally so shape_trace() and the self-test run in ANY
# environment. The architecture needs torch; the shape arithmetic does not.
try:
    import torch
    import torch.nn as nn
    import torch.nn.functional as F
    _HAS_TORCH = True
except Exception:                                     # pragma: no cover
    _HAS_TORCH = False
    nn = None


# =========================================================================== #
# 1. Shape arithmetic -- runs without torch
# =========================================================================== #
def _conv_out(i, k, s=1, p=0, d=1):
    return (i + 2 * p - d * (k - 1) - 1) // s + 1


def _deconv_out(i, k, s=1, p=0, op=0, d=1):
    return (i - 1) * s - 2 * p + d * (k - 1) + op + 1


def shape_trace(size: int = 224, verbose: bool = False) -> int:
    """Trace the spatial size through the network analytically.

    Every target map, every quality arg-max and every grasp decode assumes the
    output grid lines up with the input grid exactly. Returns the final size so
    a caller can assert it equals `size`.
    """
    h = size
    steps = [("input", h)]
    h = _conv_out(h, 9, 1, 4); steps.append(("conv1  k9 s1 p4", h))
    h = _conv_out(h, 4, 2, 1); steps.append(("conv2  k4 s2 p1", h))
    h = _conv_out(h, 4, 2, 1); steps.append(("conv3  k4 s2 p1", h))
    steps.append(("residual x5", h))
    h = _deconv_out(h, 4, 2, 1); steps.append(("deconv1 k4 s2 p1", h))
    h = _deconv_out(h, 4, 2, 1); steps.append(("deconv2 k4 s2 p1", h))
    h = _conv_out(h, 9, 1, 4); steps.append(("deconv3 k9 s1 p4", h))
    h = _conv_out(h, 1, 1, 0); steps.append(("heads   k1", h))
    if verbose:
        for name, v in steps:
            print(f"    {name:18s} {v}")
    return h


# =========================================================================== #
# 2. Architecture
# =========================================================================== #
if _HAS_TORCH:

    class ResidualBlock(nn.Module):
        """Standard 3x3 residual block; spatial size unchanged."""

        def __init__(self, in_ch: int, out_ch: int):
            super().__init__()
            self.conv1 = nn.Conv2d(in_ch, out_ch, 3, padding=1)
            self.bn1 = nn.BatchNorm2d(out_ch)
            self.conv2 = nn.Conv2d(out_ch, out_ch, 3, padding=1)
            self.bn2 = nn.BatchNorm2d(out_ch)

        def forward(self, x):
            y = F.relu(self.bn1(self.conv1(x)))
            y = self.bn2(self.conv2(y))
            return y + x

    class GRConvNet(nn.Module):
        """Generative Residual Convolutional Network for grasp detection.

        input_channels : 4 for RGB-D, 3 for RGB only, 1 for depth only
        channel_size   : base width (32 in the reference)
        dropout_prob   : >0 enables decoder dropout, REQUIRED for the S3 score
        n_blocks       : number of residual blocks (5 in the reference)
        """

        def __init__(self, input_channels: int = 4, channel_size: int = 32,
                     dropout_prob: float = 0.1, n_blocks: int = 5):
            super().__init__()
            c = channel_size

            self.conv1 = nn.Conv2d(input_channels, c, 9, stride=1, padding=4)
            self.bn1 = nn.BatchNorm2d(c)
            self.conv2 = nn.Conv2d(c, c * 2, 4, stride=2, padding=1)
            self.bn2 = nn.BatchNorm2d(c * 2)
            self.conv3 = nn.Conv2d(c * 2, c * 4, 4, stride=2, padding=1)
            self.bn3 = nn.BatchNorm2d(c * 4)

            self.res = nn.Sequential(*[ResidualBlock(c * 4, c * 4)
                                       for _ in range(n_blocks)])

            self.deconv1 = nn.ConvTranspose2d(c * 4, c * 2, 4, stride=2, padding=1)
            self.dbn1 = nn.BatchNorm2d(c * 2)
            self.deconv2 = nn.ConvTranspose2d(c * 2, c, 4, stride=2, padding=1)
            self.dbn2 = nn.BatchNorm2d(c)
            self.deconv3 = nn.Conv2d(c, c, 9, stride=1, padding=4)

            self.dropout_prob = dropout_prob
            self.dropout = nn.Dropout(p=dropout_prob) if dropout_prob > 0 else None

            # 1x1 pointwise heads -- see the module docstring. Do not "improve"
            # these to 2x2 or 3x3 without re-running shape_trace().
            self.pos_head = nn.Conv2d(c, 1, 1)
            self.cos_head = nn.Conv2d(c, 1, 1)
            self.sin_head = nn.Conv2d(c, 1, 1)
            self.width_head = nn.Conv2d(c, 1, 1)

            for m in self.modules():
                if isinstance(m, (nn.Conv2d, nn.ConvTranspose2d)):
                    nn.init.xavier_uniform_(m.weight, gain=1)
                    if m.bias is not None:
                        nn.init.zeros_(m.bias)

        def features(self, x):
            """Trunk up to the last decoder feature map.

            Exposed separately so the abstention gate can read a representation
            without a second forward pass.
            """
            y = F.relu(self.bn1(self.conv1(x)))
            y = F.relu(self.bn2(self.conv2(y)))
            y = F.relu(self.bn3(self.conv3(y)))
            y = self.res(y)
            y = F.relu(self.dbn1(self.deconv1(y)))
            y = F.relu(self.dbn2(self.deconv2(y)))
            y = self.deconv3(y)
            if self.dropout is not None:
                y = self.dropout(y)
            return y

        def forward(self, x):
            """Return (pos, cos, sin, width), each (B,1,H,W) at input size."""
            f = self.features(x)
            return (self.pos_head(f), self.cos_head(f),
                    self.sin_head(f), self.width_head(f))

        def compute_loss(self, x, y):
            """Smooth-L1 on all four heads, as in the reference."""
            y_pos, y_cos, y_sin, y_width = y
            pos, cos, sin, width = self(x)
            losses = {
                "p_loss": F.smooth_l1_loss(pos, y_pos),
                "cos_loss": F.smooth_l1_loss(cos, y_cos),
                "sin_loss": F.smooth_l1_loss(sin, y_sin),
                "width_loss": F.smooth_l1_loss(width, y_width),
            }
            return {"loss": sum(losses.values()), "losses": losses,
                    "pred": {"pos": pos, "cos": cos,
                             "sin": sin, "width": width}}


# =========================================================================== #
# 3. MC-dropout helpers
# =========================================================================== #
def enable_mc_dropout(model) -> int:
    """Put ONLY dropout layers into train mode; everything else stays eval.

    model.train() would also un-freeze BatchNorm running statistics, so the MC
    samples would differ for a reason unrelated to model uncertainty -- quietly
    corrupting S3. Returns the number of dropout layers activated so callers can
    assert it is non-zero.
    """
    n = 0
    model.eval()
    for m in model.modules():
        if m.__class__.__name__.startswith("Dropout"):
            m.train()
            n += 1
    return n


def post_process(pos, cos, sin, width, blur_sigma: float = 2.0):
    """Dense head outputs -> (quality, angle, width) numpy maps."""
    import numpy as np

    # NO sigmoid. The heads are trained to regress the [0,1] quality target
    # directly, so squashing at inference is a train/test mismatch. It does not
    # move the arg-max (sigmoid is monotonic) but it crushes the S1 score:
    # a raw range of 1.00 becomes 0.23 after sigmoid, gutting the very signal
    # the rejection rule thresholds on.
    q = pos.squeeze().detach().cpu().numpy()
    ang = (torch.atan2(sin, cos) / 2.0).squeeze().detach().cpu().numpy()
    w = width.squeeze().detach().cpu().numpy() * 100.0  # matches make_target_maps width_scale
    try:
        from scipy.ndimage import gaussian_filter
        q = gaussian_filter(q, blur_sigma)
        ang = gaussian_filter(ang, blur_sigma)
        w = gaussian_filter(w, blur_sigma)
    except ImportError:
        pass
    return q, ang, w


# =========================================================================== #
# 4. Self-test
# =========================================================================== #
def _selftest():
    def ok(c, m):
        print(f"  {'PASS' if c else 'FAIL'}  {m}")
        if not c:
            raise SystemExit(1)

    print("[shape trace @ 224]")
    out = shape_trace(224, verbose=True)
    ok(out == 224, f"224 input -> 224 output (got {out})")

    print("[other input sizes]")
    for size in (128, 240, 300, 320):
        o = shape_trace(size)
        ok(o == size, f"{size} -> {o}")

    print("[head kernel sanity]")
    ok(_conv_out(224, 1, 1, 0) == 224, "1x1 head preserves size")
    ok(_conv_out(224, 2, 1, 1) == 225, "2x2 p1 head would ADD a pixel (the bug)")
    ok(_conv_out(224, 2, 1, 0) == 223, "2x2 p0 head would DROP a pixel")

    print("\nALL SHAPE TESTS PASSED")
    if not _HAS_TORCH:
        print("(torch absent: architecture not instantiated, shapes verified "
              "arithmetically)")


if __name__ == "__main__":
    _selftest()
    if _HAS_TORCH:
        print("\n[live forward pass]")
        net = GRConvNet(input_channels=4, dropout_prob=0.1)
        x = torch.randn(2, 4, 224, 224)
        outs = net(x)
        print("  params:", round(sum(p.numel() for p in net.parameters()) / 1e6, 3), "M")
        print("  head shapes:", [tuple(o.shape) for o in outs])
        assert all(o.shape[-2:] == x.shape[-2:] for o in outs), \
            "heads must emit maps at input resolution"
        n_drop = enable_mc_dropout(net)
        print("  dropout layers for S3:", n_drop)
        assert n_drop > 0, "no dropout -> S3 would be identically zero"
        print("  features:", tuple(net.features(x).shape))
        print("  OK")


In [ ]:
%%writefile models_ggcnn.py
"""
models_ggcnn.py — GG-CNN, the second detector for the model-agnostic study
==========================================================================
GG-CNN (Morrison et al., "Closing the Loop for Robotic Grasping: A Real-time,
Generative Grasp Synthesis Approach"): a small fully-convolutional network that
emits the SAME four dense maps as GR-ConvNet, at input resolution:

    pos   : grasp quality
    cos   : cos(2*theta)
    sin   : sin(2*theta)
    width : gripper opening

It exists here for one reason: the paper's claim is that the risk-controlled
rejection rule works on ANY frozen detector. Demonstrating it on a SECOND,
architecturally different detector is what turns "works on GR-ConvNet" into
"model-agnostic". GG-CNN is the right choice because it is a classic grasp
baseline (reviewers know it), it is ~10x smaller so it trains in minutes, and it
is genuinely different from GR-ConvNet (no residual blocks, far fewer
parameters) -- so agreement across the two is meaningful, not a near-duplicate.

Drop-in contract
----------------
Same constructor keyword (`input_channels`), same forward signature
`(pos, cos, sin, width)` each (B,1,H,W), same `compute_loss`, and it works with
the shared `enable_mc_dropout` and `post_process` from models_grconvnet. So
train.py / extract_scores.py select it by name and need no other change.

The shape arithmetic runs without torch (like the GR-ConvNet module), so the
one bug that actually bites -- a head kernel that shifts the output by a pixel
-- is caught before any GPU time.
"""

from __future__ import annotations

try:
    import torch
    import torch.nn as nn
    import torch.nn.functional as F
    _HAS_TORCH = True
except Exception:                                     # pragma: no cover
    _HAS_TORCH = False
    nn = None


# =========================================================================== #
# 1. Shape arithmetic -- no torch needed
# =========================================================================== #
def _conv_out(i, k, s=1, p=0, d=1):
    return (i + 2 * p - d * (k - 1) - 1) // s + 1


def _deconv_out(i, k, s=1, p=0, op=0):
    return (i - 1) * s - 2 * p + (k - 1) + op + 1


def shape_trace(size: int = 224):
    """Trace H,W through the GG-CNN encoder/decoder arithmetically.

    GG-CNN's original design uses strided convs down then transposed convs up,
    with kernel/stride/padding chosen so the output returns to the input size.
    The transposed-conv output_padding is the fiddly part; this verifies the
    round trip lands exactly on `size`, with no torch.
    """
    # encoder: conv k9 s3 p3 ; conv k5 s2 p2 ; conv k3 s2 p1
    h = size
    h = _conv_out(h, 9, s=3, p=3)
    h = _conv_out(h, 5, s=2, p=2)
    h = _conv_out(h, 3, s=2, p=1)
    enc = h
    # decoder: convT k3 s2 p1 op1 ; convT k5 s2 p2 op1 ; convT k9 s3 p3 op1
    h = _deconv_out(h, 3, s=2, p=1, op=0)
    h = _deconv_out(h, 5, s=2, p=2, op=1)
    h = _deconv_out(h, 9, s=3, p=3, op=2)
    return dict(input=size, bottleneck=enc, output=h, ok=(h == size))


# =========================================================================== #
# 2. The network (needs torch)
# =========================================================================== #
if _HAS_TORCH:

    class GGCNN(nn.Module):
        """GG-CNN with the GR-ConvNet-compatible four-head output.

        dropout_prob > 0 places a dropout layer at the bottleneck so the shared
        MC-dropout score (S3) works on this detector too.
        """

        def __init__(self, input_channels: int = 4, dropout_prob: float = 0.1,
                     **kw):
            super().__init__()
            f = [32, 16, 8, 8, 16, 32]
            self.conv1 = nn.Conv2d(input_channels, f[0], 9, stride=3, padding=3)
            self.conv2 = nn.Conv2d(f[0], f[1], 5, stride=2, padding=2)
            self.conv3 = nn.Conv2d(f[1], f[2], 3, stride=2, padding=1)
            self.drop = nn.Dropout2d(p=dropout_prob)
            self.convt1 = nn.ConvTranspose2d(f[2], f[3], 3, stride=2,
                                             padding=1, output_padding=0)
            self.convt2 = nn.ConvTranspose2d(f[3], f[4], 5, stride=2,
                                             padding=2, output_padding=1)
            self.convt3 = nn.ConvTranspose2d(f[4], f[5], 9, stride=3,
                                             padding=3, output_padding=2)
            # 1x1 heads -- load-bearing, exactly as in GR-ConvNet: a 2x2 head
            # would shift every dense target by a pixel and silently hurt.
            self.pos_head = nn.Conv2d(f[5], 1, kernel_size=1)
            self.cos_head = nn.Conv2d(f[5], 1, kernel_size=1)
            self.sin_head = nn.Conv2d(f[5], 1, kernel_size=1)
            self.width_head = nn.Conv2d(f[5], 1, kernel_size=1)
            for m in self.modules():
                if isinstance(m, (nn.Conv2d, nn.ConvTranspose2d)):
                    nn.init.xavier_uniform_(m.weight, gain=1)
                    if m.bias is not None:
                        nn.init.zeros_(m.bias)

        def features(self, x):
            x = F.relu(self.conv1(x))
            x = F.relu(self.conv2(x))
            x = F.relu(self.conv3(x))
            x = self.drop(x)
            x = F.relu(self.convt1(x))
            x = F.relu(self.convt2(x))
            x = F.relu(self.convt3(x))
            return x

        def forward(self, x):
            """Return (pos, cos, sin, width), each (B,1,H,W) at input size."""
            f = self.features(x)
            return (self.pos_head(f), self.cos_head(f),
                    self.sin_head(f), self.width_head(f))

        def compute_loss(self, x, y):
            """Smooth-L1 on all four heads -- identical objective to GR-ConvNet
            so the two detectors are compared on equal footing."""
            y_pos, y_cos, y_sin, y_width = y
            pos, cos, sin, width = self(x)
            losses = {
                "p_loss": F.smooth_l1_loss(pos, y_pos),
                "cos_loss": F.smooth_l1_loss(cos, y_cos),
                "sin_loss": F.smooth_l1_loss(sin, y_sin),
                "width_loss": F.smooth_l1_loss(width, y_width),
            }
            return {"loss": sum(losses.values()), "losses": losses,
                    "pred": {"pos": pos, "cos": cos,
                             "sin": sin, "width": width}}


# =========================================================================== #
# 3. Self-test
# =========================================================================== #
def _selftest():
    def ok(c, m):
        print(f"  {'PASS' if c else 'FAIL'}  {m}")
        if not c:
            raise SystemExit(1)

    print("[shape arithmetic — no torch]")
    # The pipeline fixes input at 224; transposed-conv output_padding is tuned
    # for that size (it is inherently size-specific), so 224 is what must be
    # exact. This is the check that catches a pixel-shift head bug.
    tr = shape_trace(224)
    ok(tr["ok"], f"input 224 -> output {tr['output']} (round trip exact)")
    ok(tr["bottleneck"] < 224 // 8 + 2, "bottleneck is downsampled as expected")

    if _HAS_TORCH:
        print("[forward pass — torch present]")
        for ch in (3, 4):
            net = GGCNN(input_channels=ch, dropout_prob=0.1)
            x = torch.randn(2, ch, 224, 224)
            outs = net(x)
            ok(len(outs) == 4, f"{ch}-channel input -> four heads")
            for o, name in zip(outs, ("pos", "cos", "sin", "width")):
                ok(tuple(o.shape) == (2, 1, 224, 224),
                   f"{name} head shape {tuple(o.shape)} matches input size")
        print("[loss + drop-in compatibility]")
        net = GGCNN(input_channels=4)
        x = torch.randn(2, 4, 224, 224)
        y = tuple(torch.randn(2, 1, 224, 224) for _ in range(4))
        out = net.compute_loss(x, y)
        ok("loss" in out and out["loss"].item() >= 0, "compute_loss returns a loss")
        ok(set(out["pred"]) == {"pos", "cos", "sin", "width"},
           "pred dict matches GR-ConvNet's keys")
        # shared MC-dropout + post_process must accept this model unchanged
        from models_grconvnet import enable_mc_dropout, post_process
        n_drop = enable_mc_dropout(net)
        ok(n_drop >= 1, "enable_mc_dropout finds GG-CNN's dropout layer")
        q, ang, w = post_process(*net(x))
        ok(q.shape[-2:] == (224, 224), "shared post_process works on GG-CNN")
        # param count sanity: GG-CNN must be much smaller than GR-ConvNet
        n_params = sum(p.numel() for p in net.parameters())
        ok(n_params < 100_000, f"GG-CNN is small ({n_params} params)")
    else:
        print("[torch absent — shape arithmetic only]")

    print("\nALL GG-CNN TESTS PASSED")


if __name__ == "__main__":
    print("torch available:", _HAS_TORCH)
    _selftest()


In [ ]:
%%writefile models_unet.py
"""
models_unet.py — U-Net grasp detector, the THIRD detector
=========================================================
A compact U-Net emitting the same four dense maps (pos, cos, sin, width) as
GR-ConvNet and GG-CNN, at input resolution.

Why a U-Net specifically. The model-agnostic claim is only convincing if the
detectors span genuinely different architectures, not three variations of one.
The two existing detectors are:

    GR-ConvNet : encoder + RESIDUAL blocks + decoder (no long skips)
    GG-CNN     : plain shallow encoder/decoder (no long skips)

Neither uses SKIP CONNECTIONS. A U-Net is defined by them: each decoder stage is
concatenated with the matching encoder feature map, so gradient and spatial
detail flow along a path neither other model has. Agreement across residual,
plain, and skip designs is a real "across architectures" result.

Drop-in contract: same `input_channels` keyword, same `(pos, cos, sin, width)`
forward, same `compute_loss`, works with the shared `enable_mc_dropout` and
`post_process`. The bottleneck carries the dropout so the MC-dropout score (S3)
works here too. Shape arithmetic runs without torch.
"""

from __future__ import annotations

try:
    import torch
    import torch.nn as nn
    import torch.nn.functional as F
    _HAS_TORCH = True
except Exception:                                     # pragma: no cover
    _HAS_TORCH = False
    nn = None


# =========================================================================== #
# 1. Shape arithmetic -- no torch
# =========================================================================== #
def shape_trace(size: int = 224):
    """A symmetric U-Net with `depth` halvings returns to the input size iff the
    input is divisible by 2**depth. With depth=4 and 224 = 2**5 * 7, every
    halving is exact (224 -> 112 -> 56 -> 28 -> 14) and the decoder mirrors it
    back to 224. This checks the divisibility that makes the skip-concatenations
    line up; a mismatch would misalign the concatenated feature maps.
    """
    depth = 4
    h = size
    stages = [h]
    for _ in range(depth):
        if h % 2 != 0:
            return dict(input=size, ok=False, reason=f"{h} not divisible by 2")
        h //= 2
        stages.append(h)
    # decoder doubles back up
    for _ in range(depth):
        h *= 2
    return dict(input=size, bottleneck=stages[-1], output=h,
                stages=stages, ok=(h == size))


# =========================================================================== #
# 2. Network (needs torch)
# =========================================================================== #
if _HAS_TORCH:

    class _DoubleConv(nn.Module):
        def __init__(self, cin, cout):
            super().__init__()
            self.net = nn.Sequential(
                nn.Conv2d(cin, cout, 3, padding=1), nn.ReLU(inplace=True),
                nn.Conv2d(cout, cout, 3, padding=1), nn.ReLU(inplace=True))

        def forward(self, x):
            return self.net(x)

    class UNetGrasp(nn.Module):
        """Compact U-Net with four grasp heads.

        depth=4 downsampling stages with skip concatenations. Channel widths are
        kept small so it trains in minutes, comparable in cost to GG-CNN but
        architecturally distinct.
        """

        def __init__(self, input_channels: int = 4, base: int = 16,
                     dropout_prob: float = 0.1, **kw):
            super().__init__()
            c = [base, base * 2, base * 4, base * 8]
            self.d0 = _DoubleConv(input_channels, c[0])
            self.d1 = _DoubleConv(c[0], c[1])
            self.d2 = _DoubleConv(c[1], c[2])
            self.d3 = _DoubleConv(c[2], c[3])
            self.pool = nn.MaxPool2d(2)
            self.drop = nn.Dropout2d(p=dropout_prob)

            self.up3 = nn.ConvTranspose2d(c[3], c[2], 2, stride=2)
            self.u2 = _DoubleConv(c[3], c[2])          # c[2]+c[2] after concat
            self.up2 = nn.ConvTranspose2d(c[2], c[1], 2, stride=2)
            self.u1 = _DoubleConv(c[2], c[1])
            self.up1 = nn.ConvTranspose2d(c[1], c[0], 2, stride=2)
            self.u0 = _DoubleConv(c[1], c[0])

            self.pos_head = nn.Conv2d(c[0], 1, kernel_size=1)
            self.cos_head = nn.Conv2d(c[0], 1, kernel_size=1)
            self.sin_head = nn.Conv2d(c[0], 1, kernel_size=1)
            self.width_head = nn.Conv2d(c[0], 1, kernel_size=1)

            for m in self.modules():
                if isinstance(m, (nn.Conv2d, nn.ConvTranspose2d)):
                    nn.init.xavier_uniform_(m.weight, gain=1)
                    if m.bias is not None:
                        nn.init.zeros_(m.bias)

        def features(self, x):
            x0 = self.d0(x)                 # full res
            x1 = self.d1(self.pool(x0))     # 1/2
            x2 = self.d2(self.pool(x1))     # 1/4
            x3 = self.d3(self.pool(x2))     # 1/8 (bottleneck)
            x3 = self.drop(x3)
            # decoder with skip concatenations -- the defining U-Net feature
            y2 = self.u2(torch.cat([self.up3(x3), x2], dim=1))
            y1 = self.u1(torch.cat([self.up2(y2), x1], dim=1))
            y0 = self.u0(torch.cat([self.up1(y1), x0], dim=1))
            return y0

        def forward(self, x):
            """Return (pos, cos, sin, width), each (B,1,H,W) at input size."""
            f = self.features(x)
            return (self.pos_head(f), self.cos_head(f),
                    self.sin_head(f), self.width_head(f))

        def compute_loss(self, x, y):
            y_pos, y_cos, y_sin, y_width = y
            pos, cos, sin, width = self(x)
            losses = {
                "p_loss": F.smooth_l1_loss(pos, y_pos),
                "cos_loss": F.smooth_l1_loss(cos, y_cos),
                "sin_loss": F.smooth_l1_loss(sin, y_sin),
                "width_loss": F.smooth_l1_loss(width, y_width),
            }
            return {"loss": sum(losses.values()), "losses": losses,
                    "pred": {"pos": pos, "cos": cos,
                             "sin": sin, "width": width}}


# =========================================================================== #
# 3. Self-test
# =========================================================================== #
def _selftest():
    def ok(c, m):
        print(f"  {'PASS' if c else 'FAIL'}  {m}")
        if not c:
            raise SystemExit(1)

    print("[shape arithmetic — no torch]")
    tr = shape_trace(224)
    ok(tr["ok"], f"224 -> {tr['output']} through 4 skip stages (exact)")
    ok(tr["bottleneck"] == 14, f"bottleneck at 14x14 (got {tr['bottleneck']})")
    # a size that doesn't divide should be flagged, not silently misaligned
    bad = shape_trace(220)                    # 220/2/2/2/2 = 13.75 -> fails
    ok(not bad["ok"], "non-divisible input is flagged (skips would misalign)")

    if _HAS_TORCH:
        print("[forward pass — torch present]")
        for ch in (3, 4):
            net = UNetGrasp(input_channels=ch, dropout_prob=0.1)
            x = torch.randn(2, ch, 224, 224)
            outs = net(x)
            ok(len(outs) == 4, f"{ch}-channel -> four heads")
            for o, name in zip(outs, ("pos", "cos", "sin", "width")):
                ok(tuple(o.shape) == (2, 1, 224, 224),
                   f"{name} shape {tuple(o.shape)} matches input")
        print("[skip connections actually used]")
        net = UNetGrasp(input_channels=4)
        # the decoder DoubleConvs take concatenated channels: u2 input is
        # c[3]=128 (up3 gives c[2]=64, concat x2 c[2]=64 -> 128). Verify the
        # first conv of u2 expects 128 in-channels, proving concat is wired.
        first = net.u2.net[0]
        ok(first.in_channels == 128,
           f"decoder conv takes concatenated channels ({first.in_channels}) "
           f"-> skip connection is real")
        print("[drop-in compatibility]")
        from models_grconvnet import enable_mc_dropout, post_process
        ok(enable_mc_dropout(net) >= 1, "MC-dropout finds the bottleneck dropout")
        q, a, w = post_process(*net(torch.randn(1, 4, 224, 224)))
        ok(q.shape[-2:] == (224, 224), "shared post_process works on U-Net")
        y = tuple(torch.randn(2, 1, 224, 224) for _ in range(4))
        out = net.compute_loss(torch.randn(2, 4, 224, 224), y)
        ok("loss" in out, "compute_loss returns a loss")
        n = sum(p.numel() for p in net.parameters())
        print(f"  (U-Net params: {n:,})")
    else:
        print("[torch absent — shape arithmetic only]")

    print("\nALL U-NET TESTS PASSED")


if __name__ == "__main__":
    print("torch available:", _HAS_TORCH)
    _selftest()


In [ ]:
%%writefile models_evidential.py
"""
models/evidential.py
====================
Deep Evidential Regression (Amini et al., NeurIPS 2020) on top of GR-ConvNet,
providing the S4 score: a single-pass split of predictive uncertainty into
aleatoric and epistemic parts.

>>> WARNING -- READ BEFORE USING <<<
This is the one module worth cross-checking against your ICARCV implementation.
The NIG loss is easy to get subtly wrong (a misplaced softplus, alpha not
constrained above 1, a sign in the regulariser) in ways that still train and
still emit plausible numbers -- they are just the wrong numbers. Your ICARCV
version is validated by the fact that it produced the "evidential ranks best on
AUSE" result. If the two disagree, trust yours and delete this one.

The maths
---------
The head predicts the four parameters of a Normal-Inverse-Gamma prior over the
target's mean and variance:

    gamma  in R        predicted mean
    nu     > 0         "virtual observations" -- evidence for the mean
    alpha  > 1         evidence for the variance (must exceed 1 or the
                       variance below is undefined)
    beta   > 0         scale

    prediction   = gamma
    aleatoric    = beta / (alpha - 1)              data noise
    epistemic    = beta / (nu * (alpha - 1))       model ignorance

Epistemic carries the 1/nu factor: more evidence for the mean drives it down,
while aleatoric is unaffected by nu. That asymmetry is exactly what makes the
epistemic term the one expected to fire on out-of-distribution geometry, and it
is unit-tested below.
"""

from __future__ import annotations

import numpy as np

try:
    import torch
    import torch.nn as nn
    import torch.nn.functional as F
    _HAS_TORCH = True
except Exception:                                     # pragma: no cover
    _HAS_TORCH = False

EPS = 1e-8


# =========================================================================== #
# 1. numpy reference -- the ground truth for the maths, fully testable
# =========================================================================== #
def nig_nll_numpy(y, gamma, nu, alpha, beta):
    """Negative log-likelihood of the NIG evidential distribution."""
    from scipy.special import gammaln
    y, gamma = np.asarray(y, float), np.asarray(gamma, float)
    nu, alpha, beta = (np.asarray(v, float) for v in (nu, alpha, beta))
    omega = 2.0 * beta * (1.0 + nu)
    return (0.5 * np.log(np.pi / nu)
            - alpha * np.log(omega)
            + (alpha + 0.5) * np.log(nu * (y - gamma) ** 2 + omega)
            + gammaln(alpha) - gammaln(alpha + 0.5))


def nig_regulariser_numpy(y, gamma, nu, alpha):
    """Evidence penalty: scale the error by the total evidence.

    Pushes evidence DOWN where the prediction is wrong, which is what stops the
    network from becoming confidently incorrect.
    """
    return np.abs(np.asarray(y, float) - np.asarray(gamma, float)) * \
        (2.0 * np.asarray(nu, float) + np.asarray(alpha, float))


def nig_uncertainty_numpy(nu, alpha, beta):
    """(aleatoric, epistemic) from the NIG parameters."""
    nu, alpha, beta = (np.asarray(v, float) for v in (nu, alpha, beta))
    aleatoric = beta / (alpha - 1.0)
    epistemic = beta / (nu * (alpha - 1.0))
    return aleatoric, epistemic


# =========================================================================== #
# 2. torch head + loss
# =========================================================================== #
if _HAS_TORCH:

    class EvidentialHead(nn.Module):
        """Emits (gamma, nu, alpha, beta) maps with their constraints applied.

        softplus keeps nu and beta strictly positive; alpha additionally gets
        +1 so that alpha > 1 and the variance is defined. Forgetting that +1 is
        the single most common bug in evidential implementations -- it yields
        negative or exploding uncertainties on a subset of pixels.
        """

        def __init__(self, in_channels: int):
            super().__init__()
            # 1x1 pointwise, matching the deterministic heads. kernel_size=2
            # with padding=1 emits H+1 and misaligns the pixel grid.
            self.gamma = nn.Conv2d(in_channels, 1, 1)
            self.nu = nn.Conv2d(in_channels, 1, 1)
            self.alpha = nn.Conv2d(in_channels, 1, 1)
            self.beta = nn.Conv2d(in_channels, 1, 1)

        def forward(self, f):
            gamma = self.gamma(f)
            # clamp the evidence parameters. Without an upper bound a handful of
            # pixels drive nu/alpha up, the NLL runs to large negative values
            # (the -2.4 loss seen in training), and validation accuracy
            # oscillates as the model chases those pixels instead of the grasp.
            nu = torch.clamp(F.softplus(self.nu(f)) + EPS, max=50.0)
            alpha = torch.clamp(F.softplus(self.alpha(f)) + 1.0 + EPS, max=50.0)
            beta = torch.clamp(F.softplus(self.beta(f)) + EPS, min=EPS, max=50.0)
            return gamma, nu, alpha, beta

    def nig_nll(y, gamma, nu, alpha, beta):
        omega = 2.0 * beta * (1.0 + nu)
        return (0.5 * torch.log(np.pi / nu)
                - alpha * torch.log(omega)
                + (alpha + 0.5) * torch.log(nu * (y - gamma) ** 2 + omega)
                + torch.lgamma(alpha) - torch.lgamma(alpha + 0.5))

    def nig_regulariser(y, gamma, nu, alpha):
        return torch.abs(y - gamma) * (2.0 * nu + alpha)

    def evidential_loss(y, gamma, nu, alpha, beta, lam: float = 0.1,
                        reduction: str = "mean"):
        loss = nig_nll(y, gamma, nu, alpha, beta) + \
            lam * nig_regulariser(y, gamma, nu, alpha)
        if reduction == "mean":
            return loss.mean()
        if reduction == "sum":
            return loss.sum()
        return loss

    def nig_uncertainty(nu, alpha, beta):
        aleatoric = beta / (alpha - 1.0)
        epistemic = beta / (nu * (alpha - 1.0))
        return aleatoric, epistemic

    class EvidentialGRConvNet(nn.Module):
        """Evidential uncertainty head on a FROZEN deterministic backbone.

        This is the honest architecture for post-hoc uncertainty on a fixed
        detector, and it is what makes evidential training stable here. An
        earlier version gave the evidential model its OWN trainable backbone and
        let the NIG likelihood model the quality map directly; the likelihood
        then minimised itself by becoming very confident about the vast zero
        background, gamma collapsed toward 0 everywhere, the quality map lost its
        peak, and validation sat at 0% while the loss ran to -1.4.

        Here the backbone is the trained deterministic detector, frozen. It
        supplies the grasp (the 72%-accurate detector), and the evidential head
        -- the ONLY trainable part -- attaches a Normal-Inverse-Gamma
        distribution to the quality target so S4 can be read at the detector's
        selected pixel. The detector cannot be harmed because it does not train,
        and S4 becomes a genuine post-hoc uncertainty rather than a competing,
        unstable detector.

        Pass the SAME grasp maps the detector produces through `forward`, so
        decoding at inference uses the detector's quality peak, not gamma.
        """

        def __init__(self, backbone, channel_size: int = 32, freeze: bool = True):
            super().__init__()
            self.backbone = backbone
            if freeze:
                self.backbone.eval()
                for p in self.backbone.parameters():
                    p.requires_grad_(False)
            self.evidential = EvidentialHead(channel_size)

        def forward(self, x):
            # backbone frozen -> no grad through it; features are detached so the
            # evidential head cannot backprop into (and destabilise) the detector
            if any(p.requires_grad for p in self.backbone.parameters()):
                f = self.backbone.features(x)
            else:
                with torch.no_grad():
                    f = self.backbone.features(x)
                f = f.detach()
            gamma, nu, alpha, beta = self.evidential(f)
            with torch.no_grad():
                pos = self.backbone.pos_head(f)      # detector's real quality map
                cos = self.backbone.cos_head(f)
                sin = self.backbone.sin_head(f)
                width = self.backbone.width_head(f)
            # `pos` is the map used for decoding at validation/inference; `gamma`
            # and the evidence params exist only to produce the S4 score.
            return dict(pos=pos, gamma=gamma, nu=nu, alpha=alpha, beta=beta,
                        cos=cos, sin=sin, width=width)

        def compute_loss(self, x, y, lam: float = 0.1):
            # Only the evidential head trains; it regresses the quality target
            # with the NIG likelihood. The pose heads are frozen, so no loss on
            # them. Class imbalance is handled by weighting the sparse positive
            # region up to match the background mass.
            y_pos = y[0]
            out = self(x)
            per_px = evidential_loss(y_pos, out["gamma"], out["nu"],
                                     out["alpha"], out["beta"], lam=lam,
                                     reduction="none")
            w = 1.0 + 9.0 * (y_pos > 0.1).float()     # upweight the bump 10x
            loss = (per_px * w).sum() / w.sum()
            return {"loss": loss, "losses": {"evidential": loss}, "pred": out}


# =========================================================================== #
# 3. Unit tests on the maths -- run without torch
# =========================================================================== #
def _selftest():
    def ok(c, m):
        print(f"  {'PASS' if c else 'FAIL'}  {m}")
        if not c:
            raise SystemExit(1)

    print("[NLL shape]")
    y = 1.0
    base = dict(nu=2.0, alpha=3.0, beta=1.0)
    gammas = np.linspace(-2, 4, 601)
    nlls = np.array([nig_nll_numpy(y, g, **base) for g in gammas])
    ok(np.isfinite(nlls).all(), "NLL finite over a wide range of gamma")
    ok(abs(gammas[np.argmin(nlls)] - y) < 1e-2,
       f"NLL minimised at gamma == y  [argmin={gammas[np.argmin(nlls)]:.3f}]")

    print("[uncertainty decomposition]")
    al, ep = nig_uncertainty_numpy(nu=2.0, alpha=3.0, beta=1.0)
    ok(abs(al - 1.0 / 2.0) < 1e-12, "aleatoric == beta/(alpha-1)")
    ok(abs(ep - 1.0 / 4.0) < 1e-12, "epistemic == beta/(nu(alpha-1))")

    nus = np.array([0.1, 1.0, 10.0, 100.0])
    als, eps = nig_uncertainty_numpy(nus, 3.0, 1.0)
    ok(np.all(np.diff(eps) < 0), "epistemic DECREASES as evidence nu grows")
    # aleatoric = beta/(alpha-1) has no nu term at all, so it broadcasts to a
    # single value no matter how many nu values are passed. That IS the property
    # under test, so assert it after an explicit broadcast rather than indexing.
    als_b = np.broadcast_to(np.asarray(als), nus.shape)
    ok(np.allclose(als_b, als_b.flat[0]), "aleatoric is INDEPENDENT of nu")

    alphas = np.array([1.5, 3.0, 11.0])
    als2, _ = nig_uncertainty_numpy(2.0, alphas, 1.0)
    ok(np.all(np.diff(als2) < 0), "aleatoric decreases as alpha grows")

    print("[regulariser]")
    ok(nig_regulariser_numpy(1.0, 1.0, 2.0, 3.0) == 0.0,
       "regulariser == 0 when the prediction is exact")
    r_small = nig_regulariser_numpy(1.0, 0.5, 2.0, 3.0)
    r_big = nig_regulariser_numpy(1.0, 0.5, 20.0, 3.0)
    ok(r_big > r_small, "more evidence on a WRONG prediction costs more")
    ok(nig_regulariser_numpy(1.0, 2.0, 2.0, 3.0) ==
       nig_regulariser_numpy(1.0, 0.0, 2.0, 3.0),
       "regulariser is symmetric in the error")

    print("[constraint sanity]")
    softplus = lambda z: np.log1p(np.exp(-np.abs(z))) + np.maximum(z, 0)
    raw = np.array([-50.0, -5.0, 0.0, 5.0, 50.0])
    ok(np.all(softplus(raw) + EPS > 0), "softplus keeps nu, beta > 0")
    ok(np.all(softplus(raw) + 1.0 + EPS > 1.0),
       "softplus + 1 keeps alpha > 1 (variance defined)")
    al3, ep3 = nig_uncertainty_numpy(softplus(raw) + EPS,
                                     softplus(raw) + 1.0 + EPS, 1.0)
    ok(np.all(np.isfinite(al3)) and np.all(al3 > 0),
       "aleatoric finite and positive at extreme logits")
    ok(np.all(np.isfinite(ep3)) and np.all(ep3 > 0),
       "epistemic finite and positive at extreme logits")

    print("\nALL EVIDENTIAL MATH TESTS PASSED")
    print("Still cross-check against your ICARCV implementation before trusting it.")


if __name__ == "__main__":
    print("torch available:", _HAS_TORCH)
    _selftest()


In [ ]:
%%writefile metrics_grasp.py
"""
metrics/grasp.py
================
The rectangle metric that defines "success" for the whole paper.

A predicted grasp counts as correct when, against ANY ground-truth rectangle:
    * the rotated-rectangle IoU exceeds 0.25, AND
    * the gripper-plate angle differs by less than 30 degrees.

Two details are easy to get wrong and both are tested at the bottom:

1. GRIPPER ANGLE IS pi-PERIODIC. A parallel-jaw gripper at 0 and at 180 degrees
   is the SAME grasp. Any angle comparison must fold by pi, not 2*pi. Getting
   this wrong silently marks correct grasps as failures, which would corrupt
   every risk number in the paper.

2. THE POLYGONS ARE ROTATED, so an axis-aligned IoU is wrong. We clip one
   convex polygon against the other (Sutherland-Hodgman) and use the shoelace
   area. No shapely dependency, so the metric is verifiable in isolation.

This module is pure numpy on purpose: the definition of success should not
depend on torch, a GPU, or a model being loadable.
"""

from __future__ import annotations

import numpy as np

EPS = 1e-12
DEFAULT_IOU = 0.25
DEFAULT_ANGLE_DEG = 30.0


# --------------------------------------------------------------------------- #
# Representations
# --------------------------------------------------------------------------- #
def corners_to_params(rect: np.ndarray):
    """(4,2) corner array in (x, y) order -> (cx, cy, theta, w, h).

    `theta` is the direction of the edge from corner 0 to corner 1, which for
    the Cornell annotation convention is a gripper-plate edge.
    """
    rect = np.asarray(rect, dtype=float)
    cx, cy = rect[:, 0].mean(), rect[:, 1].mean()
    dx, dy = rect[1, 0] - rect[0, 0], rect[1, 1] - rect[0, 1]
    theta = np.arctan2(dy, dx)
    w = np.hypot(dx, dy)
    dx2, dy2 = rect[2, 0] - rect[1, 0], rect[2, 1] - rect[1, 1]
    h = np.hypot(dx2, dy2)
    return float(cx), float(cy), float(theta), float(w), float(h)


def params_to_corners(cx, cy, theta, w, h) -> np.ndarray:
    """(cx, cy, theta, w, h) -> (4,2) corner array in (x, y) order."""
    c, s = np.cos(theta), np.sin(theta)
    base = np.array([[-w / 2, -h / 2], [w / 2, -h / 2],
                     [w / 2, h / 2], [-w / 2, h / 2]], dtype=float)
    R = np.array([[c, -s], [s, c]], dtype=float)
    return base @ R.T + np.array([cx, cy], dtype=float)


# --------------------------------------------------------------------------- #
# Convex polygon geometry
# --------------------------------------------------------------------------- #
def polygon_area(poly: np.ndarray) -> float:
    """Shoelace area of a simple polygon; sign-independent."""
    poly = np.asarray(poly, dtype=float)
    if len(poly) < 3:
        return 0.0
    x, y = poly[:, 0], poly[:, 1]
    return float(abs(np.dot(x, np.roll(y, -1)) - np.dot(y, np.roll(x, -1))) / 2.0)


def _orient(poly: np.ndarray) -> np.ndarray:
    """Force counter-clockwise winding so the clip half-plane test is consistent."""
    poly = np.asarray(poly, dtype=float)
    x, y = poly[:, 0], poly[:, 1]
    signed = np.dot(x, np.roll(y, -1)) - np.dot(y, np.roll(x, -1))
    return poly[::-1] if signed < 0 else poly


def convex_intersection(poly_a: np.ndarray, poly_b: np.ndarray) -> np.ndarray:
    """Sutherland-Hodgman clip of convex poly_a against convex poly_b."""
    out = _orient(poly_a)
    clip = _orient(poly_b)
    n_clip = len(clip)

    for i in range(n_clip):
        if len(out) == 0:
            return np.zeros((0, 2))
        a, b = clip[i], clip[(i + 1) % n_clip]
        edge = b - a

        def inside(p):
            # left of the directed edge a->b, for CCW winding
            return edge[0] * (p[1] - a[1]) - edge[1] * (p[0] - a[0]) >= -1e-9

        new = []
        m = len(out)
        for j in range(m):
            cur, nxt = out[j], out[(j + 1) % m]
            cur_in, nxt_in = inside(cur), inside(nxt)
            if cur_in:
                new.append(cur)
            if cur_in != nxt_in:
                d = nxt - cur
                denom = edge[0] * d[1] - edge[1] * d[0]
                if abs(denom) > 1e-12:
                    t = (edge[0] * (cur[1] - a[1]) -
                         edge[1] * (cur[0] - a[0])) / -denom
                    new.append(cur + t * d)
        out = np.array(new, dtype=float) if new else np.zeros((0, 2))
    return out


def rect_iou(rect_a: np.ndarray, rect_b: np.ndarray) -> float:
    """IoU of two rotated rectangles given as (4,2) corner arrays."""
    a = np.asarray(rect_a, dtype=float)
    b = np.asarray(rect_b, dtype=float)
    inter = polygon_area(convex_intersection(a, b))
    union = polygon_area(a) + polygon_area(b) - inter
    return float(inter / union) if union > EPS else 0.0


# --------------------------------------------------------------------------- #
# Angle handling
# --------------------------------------------------------------------------- #
def angle_diff(a: float, b: float) -> float:
    """Smallest angle between two gripper orientations, folded by pi.

    A parallel-jaw gripper at theta and theta+pi is the same physical grasp, so
    the difference must fold by pi. Result is in [0, pi/2].
    """
    d = abs(float(a) - float(b)) % np.pi
    return float(min(d, np.pi - d))


# --------------------------------------------------------------------------- #
# Success criterion
# --------------------------------------------------------------------------- #
def grasp_success(pred, gts, iou_thresh: float = DEFAULT_IOU,
                  angle_thresh_deg: float = DEFAULT_ANGLE_DEG) -> int:
    """1 if `pred` matches ANY ground-truth grasp, else 0.

    pred : dict with x, y, theta (a decoded grasp); (4,2) corners; or
           (cx, cy, theta, w, h).
    gts  : (N,4,2) array or list of (4,2) ground-truth corner arrays.

    Cornell's rectangle metric compares CENTER, ANGLE and rectangle overlap. The
    subtlety that broke an earlier version: a predicted grasp does not carry a
    trustworthy jaw HEIGHT (the model regresses opening width, not plate
    length), so rebuilding the predicted rectangle from a *guessed* aspect and
    demanding full-rectangle IoU caps the score no matter how good the model is.

    The fix: give the predicted grasp the SAME height as the ground-truth
    rectangle it is being tested against, then require the centers to be close,
    the angles within threshold, and the widths comparable. This is exactly the
    information the metric is entitled to use -- the annotation supplies the
    height -- and it removes the phantom miss.
    """
    px, py, p_theta, p_w = _as_center_angle_width(pred)
    if px is None:
        return 0
    gts = np.asarray(gts, dtype=float)
    if gts.size == 0:
        return 0
    if gts.ndim == 2:
        gts = gts[None]

    ang_thresh = np.deg2rad(angle_thresh_deg)
    for g in gts:
        gcx, gcy, g_theta, g_w, g_h = corners_to_params(g)
        # width from corners_to_params is the edge0->1 length; the physical jaw
        # opening is the SHORTER side, so use min(w,h) for both.
        g_open = min(g_w, g_h)
        if angle_diff(p_theta, g_theta) >= ang_thresh:
            continue
        # build both rectangles with the SAME height (the GT plate length) so
        # aspect can never cause a phantom miss; vary only center/angle/width.
        g_plate = max(g_w, g_h)
        pred_rect = params_to_corners(px, py, p_theta, max(p_w, 1.0), g_plate)
        gt_rect = params_to_corners(gcx, gcy, g_theta, g_open, g_plate)
        if rect_iou(pred_rect, gt_rect) > iou_thresh:
            return 1
    return 0


def _as_center_angle_width(pred):
    """Normalise a predicted grasp to (x, y, theta, width)."""
    if isinstance(pred, dict):
        x = pred.get("x", pred.get("cx"))
        y = pred.get("y", pred.get("cy"))
        th = pred.get("theta", pred.get("angle", 0.0))
        w = pred.get("width", pred.get("w", 0.0))
        if x is None or y is None:
            return None, None, None, None
        return float(x), float(y), float(th), float(w)
    arr = np.asarray(pred, dtype=float)
    if arr.shape == (4, 2):
        cx, cy, th, w, h = corners_to_params(arr)
        return cx, cy, th, min(w, h)
    if arr.shape == (5,):
        cx, cy, th, w, h = arr
        return float(cx), float(cy), float(th), float(min(w, h))
    raise ValueError(f"unsupported grasp representation with shape {arr.shape}")


def _as_rect_and_theta(pred):
    """Accept several grasp representations and normalise to (corners, theta)."""
    if isinstance(pred, dict):
        if "corners" in pred:
            r = np.asarray(pred["corners"], dtype=float)
            return r, corners_to_params(r)[2]
        cx = pred.get("x", pred.get("cx"))
        cy = pred.get("y", pred.get("cy"))
        th = pred.get("theta", pred.get("angle", 0.0))
        w = pred.get("width", pred.get("w", 0.0))
        h = pred.get("height", pred.get("h", w * 0.5))
        if cx is None or cy is None or not w:
            return None, 0.0
        return params_to_corners(cx, cy, th, w, h), float(th)
    arr = np.asarray(pred, dtype=float)
    if arr.shape == (4, 2):
        return arr, corners_to_params(arr)[2]
    if arr.shape == (5,):
        cx, cy, th, w, h = arr
        return params_to_corners(cx, cy, th, w, h), float(th)
    raise ValueError(f"unsupported grasp representation with shape {arr.shape}")


def evaluate_batch(preds, gts_list, **kw) -> list[int]:
    """Per-sample success indicators for a batch."""
    return [grasp_success(p, g, **kw) for p, g in zip(preds, gts_list)]


# --------------------------------------------------------------------------- #
# Unit tests
# --------------------------------------------------------------------------- #
def _selftest():
    ok = lambda c, m: print(f"  {'PASS' if c else 'FAIL'}  {m}") or (c or exit(1))

    print("[geometry]")
    sq = params_to_corners(0, 0, 0, 2, 2)
    ok(abs(polygon_area(sq) - 4.0) < 1e-9, "area of 2x2 square == 4")
    ok(abs(rect_iou(sq, sq) - 1.0) < 1e-9, "IoU(identical) == 1")

    far = params_to_corners(100, 100, 0, 2, 2)
    ok(rect_iou(sq, far) == 0.0, "IoU(disjoint) == 0")

    # two unit squares offset by half: inter=0.5, union=1.5 -> 1/3
    a = params_to_corners(0, 0, 0, 1, 1)
    b = params_to_corners(0.5, 0, 0, 1, 1)
    ok(abs(rect_iou(a, b) - 1.0 / 3.0) < 1e-6, "IoU(half-overlap) == 1/3")

    # a square rotated 90 deg about its centre is itself
    r90 = params_to_corners(0, 0, np.pi / 2, 2, 2)
    ok(abs(rect_iou(sq, r90) - 1.0) < 1e-6, "IoU(square rotated 90) == 1")

    # 45-degree rotated square: known intersection area 8*(sqrt2 - 1)
    big = params_to_corners(0, 0, 0, 2, 2)
    rot45 = params_to_corners(0, 0, np.pi / 4, 2, 2)
    inter = polygon_area(convex_intersection(big, rot45))
    ok(abs(inter - 8 * (np.sqrt(2) - 1)) < 1e-6,
       f"45-deg overlap area == 8(sqrt2-1)  [{inter:.6f}]")

    print("[angle folding]")
    ok(angle_diff(0.0, np.pi) < 1e-9, "0 and pi are the SAME gripper angle")
    ok(angle_diff(0.0, 2 * np.pi) < 1e-9, "0 and 2pi identical")
    ok(abs(angle_diff(0.0, np.pi / 2) - np.pi / 2) < 1e-9, "0 vs 90deg == 90deg")
    ok(abs(angle_diff(np.deg2rad(10), np.deg2rad(170)) - np.deg2rad(20)) < 1e-9,
       "10deg vs 170deg == 20deg apart (folded)")
    ok(angle_diff(0, np.deg2rad(179)) <= np.pi / 2 + 1e-9, "folded diff <= 90deg")

    print("[success criterion]")
    # GT plate length 40, jaw opening 20, at origin, angle 0
    gt = params_to_corners(50, 50, 0.0, 40, 20)   # w=40 (plate), h=20 (opening)
    # a decoded grasp is a dict with x,y,theta,width(=opening)
    good = dict(x=50, y=50, theta=0.0, width=20)
    ok(grasp_success(good, [gt]) == 1, "matching decoded grasp succeeds")

    flipped = dict(x=50, y=50, theta=np.pi, width=20)
    ok(grasp_success(flipped, [gt]) == 1,
       "grasp flipped by pi still succeeds (pi-periodicity)")

    rot = dict(x=50, y=50, theta=np.pi / 4, width=20)
    ok(grasp_success(rot, [gt]) == 0, "45deg off rejected by the angle gate")

    shifted = dict(x=200, y=200, theta=0.0, width=20)
    ok(grasp_success(shifted, [gt]) == 0, "distant grasp rejected by IoU gate")

    near = dict(x=50, y=50, theta=np.deg2rad(25), width=20)
    ok(grasp_success(near, [gt]) == 1, "25deg off, aligned -> succeeds")
    far_ang = dict(x=50, y=50, theta=np.deg2rad(35), width=20)
    ok(grasp_success(far_ang, [gt]) == 0, "35deg off fails (threshold 30)")

    # slightly wrong opening still succeeds; wildly wrong opening fails
    ok(grasp_success(dict(x=50, y=50, theta=0.0, width=26), [gt]) == 1,
       "opening off by 30% still overlaps enough")
    ok(grasp_success(dict(x=50, y=50, theta=0.0, width=90), [gt]) == 0,
       "opening 4.5x too wide fails")

    gts = [params_to_corners(10, 10, 0, 40, 20), gt]
    ok(grasp_success(good, gts) == 1, "matches any ground-truth in the set")
    ok(grasp_success(good, np.zeros((0, 4, 2))) == 0, "empty GT set -> failure")

    print("[representations]")
    ok(grasp_success(gt, [gt]) == 1, "(4,2) corner prediction accepted")
    ok(grasp_success(np.array([50, 50, 0.0, 20, 40]), [gt]) == 1,
       "(cx,cy,theta,w,h) tuple accepted")
    ok(evaluate_batch([good, shifted], [[gt], [gt]]) == [1, 0], "batch evaluation")

    print("\nALL GRASP-METRIC TESTS PASSED")


if __name__ == "__main__":
    _selftest()


In [ ]:
%%writefile train.py
"""
train.py
========
Trains the frozen-backbone-to-be on Cornell. Two methods only:

    deterministic  -> supplies S1, S2, S5 (quality-map scores) and, with
                      dropout re-enabled at inference, S3
    evidential     -> supplies S4 (aleatoric / epistemic split)

Ensemble and heteroscedastic are deliberately NOT here. The FISAT paper does not
use them, and dropping them roughly halves the retraining time -- which is the
whole reason for moving to Cornell.

Usage
-----
    python train.py --config configs/cornell.yaml --method deterministic
    python train.py --config configs/cornell.yaml --method evidential

    # verify the pipeline end-to-end on random tensors, no dataset needed:
    python train.py --smoke

Checkpoints land in runs/<method>/{best.pt,last.pt}. BACK THEM UP THE MOMENT
TRAINING ENDS -- losing these is what forced this rebuild in the first place.
"""

from __future__ import annotations

import argparse
import json
import os
import time

import numpy as np

try:
    import torch
    from torch.utils.data import DataLoader
    _HAS_TORCH = True
except Exception:                                     # pragma: no cover
    _HAS_TORCH = False


# =========================================================================== #
# 1. Ground-truth rasterisation: grasp rectangles -> dense target maps
# =========================================================================== #
def _rect_mask(corners: np.ndarray, size: int, shrink: float = 1.0/3.0):
    """Boolean mask of a rotated rectangle, shrunk about its centre.

    The reference implementations paint only the INNER third along the gripper
    axis. Painting the full rectangle would label the open jaw span as graspable
    and blur the quality peak, so the shrink is not cosmetic.
    """
    from metrics_grasp import corners_to_params

    cx, cy, theta, w, h = corners_to_params(corners)
    # Fold theta into [0, pi). A rectangle has 180-degree rotational symmetry, so
    # theta and theta+pi describe the SAME region -- but they produce
    # sign-flipped rotated coordinates, and the boundary test below then
    # includes or excludes a different set of edge pixels (measured: 231 vs 217
    # for the same rectangle). Canonicalising makes rasterisation depend on the
    # grasp, not on how the annotation happened to order its corners.
    theta = theta % np.pi
    # Shrink ONLY the jaw-opening direction (h), never both. Shrinking both
    # multiplies the area by 1/9 instead of 1/3, and the positive region is
    # already only ~0.1% of the map: measured on a typical Cornell rectangle,
    # shrinking both leaves 45 positive pixels where the reference leaves 131.
    # With a regression loss that imbalance collapses every head to zero, and a
    # zero-width prediction has zero area, so IoU is 0 on every single sample --
    # which is exactly the 0.00% validation accuracy this produced.
    w_s, h_s = w, h * shrink
    yy, xx = np.mgrid[0:size, 0:size]
    dx, dy = xx - cx, yy - cy
    c, s = np.cos(-theta), np.sin(-theta)
    xr = dx * c - dy * s
    yr = dx * s + dy * c
    return (np.abs(xr) <= w_s / 2.0) & (np.abs(yr) <= h_s / 2.0)


def make_target_maps(gts: np.ndarray, size: int = 224, width_scale: float = 100.0):
    """(N,4,2) ground-truth rectangles -> (pos, cos, sin, width) maps.

    The angle is stored as cos(2*theta) / sin(2*theta), NOT cos/sin of theta.
    Doubling folds theta and theta+pi onto the same point, matching the physical
    fact that a parallel-jaw gripper flipped by 180 degrees is the same grasp.
    Storing the raw angle instead would force the network to learn a
    discontinuity that does not exist. Round-tripping is unit-tested below.
    """
    from metrics_grasp import corners_to_params

    pos = np.zeros((size, size), np.float32)
    cos_m = np.zeros((size, size), np.float32)
    sin_m = np.zeros((size, size), np.float32)
    wid = np.zeros((size, size), np.float32)

    gts = np.asarray(gts, dtype=float)
    if gts.size == 0:
        return pos, cos_m, sin_m, wid
    if gts.ndim == 2:
        gts = gts[None]

    yy, xx = np.mgrid[0:size, 0:size]
    for rect in gts:
        cx, cy, theta, w, h = corners_to_params(rect)
        opening = min(w, h)                 # jaw opening = shorter side
        plate = max(w, h)                   # gripper-plate length
        m = _rect_mask(rect, size)          # region that carries angle/width
        if not m.any():
            continue
        # QUALITY: a Gaussian bump centered on the grasp, not a flat plateau.
        # sigma scales with the grasp so small grasps stay sharp. The arg-max of
        # a Gaussian is its center, which is what decode_grasp reads, so the
        # decoded grasp lands where the annotation is. Take the max across
        # overlapping grasps so multiple annotations each keep a peak.
        sigma = max(plate / 6.0, 3.0)
        bump = np.exp(-(((xx - cx) ** 2 + (yy - cy) ** 2) / (2.0 * sigma ** 2)))
        np.maximum(pos, bump, out=pos)
        # ANGLE / WIDTH: filled over the region so the heads have a target with
        # spatial support, but ONLY where this grasp is the closest peak. Using
        # the mask is fine here because these maps are read at the quality
        # arg-max, which the Gaussian pins to the center.
        cos_m[m] = np.cos(2.0 * theta)
        sin_m[m] = np.sin(2.0 * theta)
        wid[m] = np.clip(opening, 0, width_scale) / width_scale
    return pos, cos_m, sin_m, wid


def decode_angle(cos_m, sin_m):
    """Inverse of the doubled-angle encoding: theta = atan2(sin, cos) / 2."""
    return np.arctan2(sin_m, cos_m) / 2.0


# =========================================================================== #
# 2. Decoding a grasp from predicted maps
# =========================================================================== #
def decode_grasp(quality, angle, width, width_scale: float = 100.0):
    """Read the grasp at the quality arg-max."""
    # `width` arrives already rescaled to pixels by post_process(). The earlier
    # version guessed with `if width.max() > 1.5 else width_scale`, which
    # silently multiplied by 150 a second time whenever the width head was still
    # near zero -- exactly when the guess is least reliable.
    idx = np.unravel_index(np.argmax(quality), quality.shape)
    return dict(y=int(idx[0]), x=int(idx[1]),
                theta=float(angle[idx]),
                width=float(width[idx]),
                quality=float(quality[idx]))


def grasp_to_corners(g, jaw_frac: float = 0.5):
    """Rectangle corners for a decoded grasp, for DRAWING ONLY.

    Do NOT feed this to grasp_success. grasp_success takes the decoded grasp
    dict directly and pulls the plate length from the ground-truth rectangle it
    compares against; routing through this helper instead rebuilds the rectangle
    with a GUESSED height (width*jaw_frac), reintroducing the aspect-mismatch
    that pins the success metric to 0. This function exists only so figures can
    draw a plausible rectangle for a standalone prediction.
    """
    from metrics_grasp import params_to_corners
    return params_to_corners(g["x"], g["y"], g["theta"],
                             g["width"], g["width"] * jaw_frac)


# =========================================================================== #
# 3. Training
# =========================================================================== #
def collate_with_targets(batch, size=224):
    """Attach rasterised target maps to a Cornell batch."""
    import torch
    images = torch.stack([b["image"] for b in batch], 0)
    pos, cs, sn, wd = [], [], [], []
    for b in batch:
        p, c, s, w = make_target_maps(b["gt_grasps"], size)
        pos.append(p); cs.append(c); sn.append(s); wd.append(w)
    to_t = lambda arr: torch.from_numpy(np.stack(arr, 0)[:, None])
    return {"image": images,
            "target": (to_t(pos), to_t(cs), to_t(sn), to_t(wd)),
            "gt_grasps": [b["gt_grasps"] for b in batch],
            "obj_id": [b["obj_id"] for b in batch],
            "idx": [b["idx"] for b in batch]}


@torch.no_grad() if _HAS_TORCH else (lambda f: f)
def validate(model, loader, device, method):
    """Validation success rate under the rectangle metric.

    We early-stop on the GRASP METRIC, not on the loss. A lower smooth-L1 on the
    dense maps does not reliably mean a better grasp at the arg-max, and the
    metric is what the paper reports.
    """
    from metrics_grasp import grasp_success
    from models_grconvnet import post_process

    model.eval()
    n_ok = n_tot = 0
    for batch in loader:
        x = batch["image"].to(device)
        out = model(x)
        for i in range(x.shape[0]):
            if method == "evidential":
                # decode from the FROZEN detector's real quality map (pos), not
                # gamma. gamma exists only to produce the S4 uncertainty; the
                # grasp itself comes from the 72%-accurate detector underneath.
                q = out["pos"][i:i+1]
                c, s, w = out["cos"][i:i+1], out["sin"][i:i+1], out["width"][i:i+1]
            else:
                q, c, s, w = (o[i:i+1] for o in out)
            qm, am, wm = post_process(q, c, s, w)
            g = decode_grasp(qm, am, wm)
            n_ok += grasp_success(g, batch["gt_grasps"][i])
            n_tot += 1
    return n_ok / max(n_tot, 1)


def make_dataset_factory(cfg):
    """Return a `split -> Dataset` builder chosen by cfg['data']['name'].

    Both loaders share the same positional signature, so the only per-dataset
    difference is the class and a couple of Jacquard-only keyword args. Keeping
    this in one place is what lets train.py / extract_scores.py stay identical
    across datasets.
    """
    name = cfg["data"].get("name", "cornell").lower()
    d = cfg["data"]
    size = d["out_size"]
    seed = cfg["train"].get("seed", 0)
    common = dict(use_depth=d.get("use_depth", True),
                  val_frac=d["val_frac"], novel_frac=d.get("novel_frac", 0.0),
                  test_frac=d["test_frac"], seed=seed,
                  split_mode=d.get("split_mode", "object"))
    if name == "jacquard":
        from data_jacquard import Jacquard
        return lambda split: Jacquard(
            d["root"], split, size, mode="real",
            depth_kind=d.get("depth_kind", "perfect"),
            max_images=d.get("max_images"), max_objects=d.get("max_objects"),
            **common)
    from data_cornell import Cornell
    return lambda split: Cornell(d["root"], split, size, mode="real", **common)


def make_model(cfg, in_ch):
    """Build the detector chosen by cfg['model']['name'] (default grconvnet).

    Both detectors share the constructor keyword and the four-head output, so
    the rest of training is identical. This is what lets the paper run the SAME
    rejection pipeline on two architectures and claim model-agnosticism.
    """
    name = cfg.get("model", {}).get("name", "grconvnet").lower()
    dropout = cfg.get("train", {}).get("dropout", 0.1)
    if name == "ggcnn":
        from models_ggcnn import GGCNN
        return GGCNN(input_channels=in_ch, dropout_prob=dropout)
    if name == "unet":
        from models_unet import UNetGrasp
        return UNetGrasp(input_channels=in_ch, dropout_prob=dropout)
    from models_grconvnet import GRConvNet
    return GRConvNet(in_ch, dropout_prob=dropout)


def train(cfg, method: str, out_dir: str = "runs"):
    import torch
    from models_grconvnet import GRConvNet

    device = cfg["train"].get("device", "cuda" if torch.cuda.is_available() else "cpu")
    size = cfg["data"]["out_size"]
    in_ch = 4 if cfg["data"].get("use_depth", True) else 3
    torch.manual_seed(cfg["train"].get("seed", 0))

    mk = make_dataset_factory(cfg)

    coll = lambda b: collate_with_targets(b, size)
    tr = DataLoader(mk("train"), batch_size=cfg["train"]["batch_size"],
                    shuffle=True, num_workers=cfg["data"].get("num_workers", 4),
                    collate_fn=coll, drop_last=True)
    va = DataLoader(mk("val"), batch_size=cfg["train"]["batch_size"],
                    shuffle=False, num_workers=cfg["data"].get("num_workers", 4),
                    collate_fn=coll)
    print(f"train={len(tr.dataset)}  val={len(va.dataset)}  device={device}")

    backbone = make_model(cfg, in_ch)
    if method == "evidential":
        # The evidential head sits on the FROZEN deterministic detector, so that
        # checkpoint must exist first. This is what keeps evidential training
        # stable: the detector is already good and cannot be harmed.
        det_ckpt = os.path.join(out_dir, "deterministic", "best.pt")
        if not os.path.exists(det_ckpt):
            raise SystemExit(
                "Train the deterministic model first: the evidential head "
                "attaches to its frozen backbone. Expected " + det_ckpt)
        state = torch.load(det_ckpt, map_location=device)
        backbone.load_state_dict(state if "state_dict" not in state
                                 else state["state_dict"])
        from models_evidential import EvidentialGRConvNet
        model = EvidentialGRConvNet(backbone, freeze=True).to(device)
    else:
        model = backbone.to(device)

    opt = torch.optim.Adam(model.parameters(), lr=cfg["train"].get("lr", 1e-3))
    run_dir = os.path.join(out_dir, method)
    os.makedirs(run_dir, exist_ok=True)

    best, bad, hist = -1.0, 0, []
    epochs = cfg["train"]["epochs"]
    patience = cfg["train"].get("patience", 12)

    for ep in range(epochs):
        model.train()
        t0, tot, nb = time.time(), 0.0, 0
        for batch in tr:
            x = batch["image"].to(device)
            y = tuple(t.to(device) for t in batch["target"])
            res = model.compute_loss(x, y)
            opt.zero_grad(); res["loss"].backward(); opt.step()
            tot += float(res["loss"].detach()); nb += 1

        acc = validate(model, va, device, method)
        hist.append(dict(epoch=ep, loss=tot / max(nb, 1), val_acc=acc))
        print(f"  ep {ep:3d}  loss={tot/max(nb,1):.4f}  val={acc*100:.2f}%  "
              f"({time.time()-t0:.0f}s)")

        torch.save(model.state_dict(), os.path.join(run_dir, "last.pt"))
        if acc > best:
            best, bad = acc, 0
            torch.save(model.state_dict(), os.path.join(run_dir, "best.pt"))
        else:
            bad += 1
            if patience > 0 and bad >= patience:
                print(f"  early stop at epoch {ep} (best {best*100:.2f}%)")
                break

    json.dump(dict(method=method, best_val=best, history=hist),
              open(os.path.join(run_dir, "history.json"), "w"), indent=2)
    print(f"\nbest val accuracy: {best*100:.2f}%   ->  {run_dir}/best.pt")
    print("BACK THIS UP TO DRIVE NOW.")
    return best


# =========================================================================== #
# 4. Tests
# =========================================================================== #
def _selftest():
    def ok(c, m):
        print(f"  {'PASS' if c else 'FAIL'}  {m}")
        if not c:
            raise SystemExit(1)

    from metrics_grasp import params_to_corners

    print("[target rasterisation]")
    size = 224
    rect = params_to_corners(112, 112, np.deg2rad(30), 60, 30)
    pos, cm, sm, wm = make_target_maps(rect[None], size)
    ok(pos.shape == (size, size), "maps are at input resolution")
    ok(0 < pos.sum() < size * size, "positive region is non-empty and not the whole image")
    ok(abs(pos.max() - 1.0) < 1e-6, "quality peak reaches 1 at the grasp center")
    # the arg-max of the quality target must sit at the grasp center, since that
    # is what decode reads back -- this is the exact property whose absence made
    # a perfectly-fit model score 0%
    peak = np.unravel_index(np.argmax(pos), pos.shape)
    ok(abs(peak[1] - 112) <= 1 and abs(peak[0] - 112) <= 1,
       f"quality arg-max is at the grasp center (got {peak[::-1]})")

    print("[doubled-angle encoding]")
    for deg in (0, 15, 30, 75, 89, 120, 179):
        th = np.deg2rad(deg)
        r = params_to_corners(112, 112, th, 60, 30)
        _, c, s, _ = make_target_maps(r[None], size)
        m = (c != 0) | (s != 0)
        dec = decode_angle(c[m].mean(), s[m].mean())
        from metrics_grasp import angle_diff
        ok(angle_diff(dec, th) < 1e-6,
           f"theta={deg:3d}deg round-trips through cos2t/sin2t")

    th0 = params_to_corners(112, 112, 0.0, 60, 30)
    thpi = params_to_corners(112, 112, np.pi, 60, 30)
    _, c0, s0, _ = make_target_maps(th0[None], size)
    _, cp, sp, _ = make_target_maps(thpi[None], size)
    ok(np.allclose(c0, cp, atol=1e-6) and np.allclose(s0, sp, atol=1e-6),
       "theta and theta+pi produce IDENTICAL targets (as they must)")

    print("[width target]")
    # plate 60, opening 30 -> stores the OPENING (min side) / 100
    r = params_to_corners(112, 112, 0.0, 60, 30)
    _, _, _, w = make_target_maps(r[None], size)
    ok(abs(w[w > 0].mean() - 0.30) < 1e-5,
       "width target = jaw opening / 100 (opening 30 -> 0.30)")

    print("[decode]")
    q = np.zeros((size, size)); q[100, 50] = 1.0
    a = np.zeros((size, size)); a[100, 50] = 0.4
    wmap = np.full((size, size), 0.3)
    g = decode_grasp(q, a, wmap)
    ok(g["y"] == 100 and g["x"] == 50, "decode finds the quality arg-max")
    ok(abs(g["theta"] - 0.4) < 1e-9, "decode reads the angle at the peak")

    print("[empty ground truth]")
    p2, _, _, _ = make_target_maps(np.zeros((0, 4, 2)), size)
    ok(p2.sum() == 0, "empty GT gives an all-zero target")

    print("[validate path round-trip]")
    # This mirrors EXACTLY what validate() does: build a perfect target, run it
    # through post_process's transforms and decode_grasp, then score it the way
    # validate scores it. It is the guard that was missing: the earlier
    # round-trip test passed the decoded grasp straight to grasp_success, while
    # validate() routed it through grasp_to_corners first, so the two disagreed
    # and a real 0%% slipped through green tests.
    from metrics_grasp import grasp_success as _gs
    try:
        from scipy.ndimage import gaussian_filter as _gf
    except ImportError:
        _gf = lambda a, s: a
    rng = np.random.default_rng(3)
    ok_n = 0
    for _ in range(40):
        cx, cy = rng.uniform(50, 174, 2)
        th = rng.uniform(-np.pi / 2, np.pi / 2)
        gt = params_to_corners(cx, cy, th, rng.uniform(50, 90), rng.uniform(20, 45))
        p, cm, sm, wd = make_target_maps(gt[None], size)
        ang = np.arctan2(sm, cm) / 2.0
        g = decode_grasp(_gf(p.astype(float), 2.0), _gf(ang, 2.0),
                         _gf(wd * 100.0, 2.0))
        ok_n += _gs(g, [gt])          # EXACTLY validate()'s scoring call
    ok(ok_n >= 38,
       f"perfect target scores via the validate() path ({ok_n}/40 succeed)")

    print("\nALL TRAIN-PIPELINE TESTS PASSED")


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--config", default="configs/cornell.yaml")
    ap.add_argument("--method", default="deterministic",
                    choices=["deterministic", "evidential"])
    ap.add_argument("--out", default="runs")
    ap.add_argument("--smoke", action="store_true",
                    help="run the offline self-tests and exit")
    args = ap.parse_args()

    if args.smoke:
        _selftest()
        return

    import yaml
    cfg = yaml.safe_load(open(args.config))
    train(cfg, args.method, args.out)


if __name__ == "__main__":
    main()


In [ ]:
%%writefile extract_scores.py
"""
extract_scores.py
=================
Runs the FROZEN detectors over a Cornell split ONCE and caches everything the
rest of the paper needs into a single .npz:

    idx      (N,)     dataset index, so figures can find the image again
    obj_id   (N,)     folder-level group id, for object-aware splitting
    success  (N,)     1 if the predicted grasp passes the rectangle metric
    s_conf   (N,)     S1  1 - peak quality
    s_ent    (N,)     S2  spatial entropy of the quality map
    s_mc     (N,)     S3  MC-dropout pose dispersion
    s_topk   (N,)     S5  top-k grasp disagreement
    s_ev_al  (N,)     S4  evidential aleatoric      (if --ev-ckpt given)
    s_ev_ep  (N,)     S4  evidential epistemic      (if --ev-ckpt given)
    feat     (N,D)    pooled features for the learned abstention gate

Convention for every score: LARGER = LESS TRUSTWORTHY. Accept iff s <= tau.

This is the ONLY expensive step in the project. Everything downstream reads the
.npz and runs in seconds, so BACK THE .npz UP as soon as it exists.

    python extract_scores.py --config configs/cornell.yaml \
        --ckpt runs/deterministic/best.pt \
        --ev-ckpt runs/evidential/best.pt \
        --split test --out scores.npz
"""

from __future__ import annotations

import argparse

import numpy as np
import torch

from train import make_dataset_factory
from metrics_grasp import grasp_success
from models_grconvnet import GRConvNet, enable_mc_dropout, post_process
from train import make_model
from train import decode_grasp, grasp_to_corners

EPS = 1e-12


# --------------------------------------------------------------------------- #
# Model loading
# --------------------------------------------------------------------------- #
def _load_state(net, ckpt, device):
    state = torch.load(ckpt, map_location=device)
    if isinstance(state, dict) and "state_dict" in state:
        state = state["state_dict"]
    net.load_state_dict(state)
    net.to(device).eval()
    for p in net.parameters():
        p.requires_grad_(False)
    return net


def load_backbone(ckpt, in_ch, device, dropout=0.1, cfg=None):
    return _load_state(make_model(cfg or {}, in_ch), ckpt, device)


def load_evidential(ckpt, in_ch, device, dropout=0.1, det_ckpt=None, cfg=None):
    from models_evidential import EvidentialGRConvNet
    backbone = make_model(cfg or {}, in_ch)
    if det_ckpt:
        st = torch.load(det_ckpt, map_location=device)
        backbone.load_state_dict(st if "state_dict" not in st else st["state_dict"])
    net = EvidentialGRConvNet(backbone, freeze=True)
    # the saved evidential checkpoint contains the full module (frozen backbone
    # weights included), so loading it restores everything
    return _load_state(net, ckpt, device)


# --------------------------------------------------------------------------- #
# S1 / S2 — quality-map scores (one forward pass)
# --------------------------------------------------------------------------- #
def score_conf(q) -> float:
    """S1: 1 - peak quality. The cheapest score, and the baseline every other
    one must beat to justify its extra compute."""
    return float(1.0 - q.max())


def score_entropy(q, floor_ratio: float = 0.5) -> float:
    """S2: spatial entropy of the quality map above a RELATIVE floor.

    The floor is essential. Without it the near-zero background dominates the
    normalised sum and the entropy becomes a proxy for how much of the frame is
    object, not for grasp ambiguity.
    """
    m = q.copy()
    m[m < floor_ratio * m.max()] = 0.0
    s = m.sum()
    if s <= EPS:
        return float(np.log(m.size))
    p = m.ravel() / s
    p = p[p > 0]
    return float(-(p * np.log(p)).sum())


# --------------------------------------------------------------------------- #
# S5 — top-k grasp disagreement (one forward pass)
# --------------------------------------------------------------------------- #
def score_topk_disagreement(q, ang, w, k: int = 5, nms_radius: int = 20) -> float:
    """Quality-weighted mean pairwise pose distance among top-k local maxima.

    Catches scene ambiguity: several distinct, similarly-good, mutually
    incompatible grasps. Neither peak confidence nor per-pixel variance sees
    this, which is why it is the cheapest useful complement to S1.
    """
    try:
        from skimage.feature import peak_local_max
        peaks = peak_local_max(q, min_distance=nms_radius,
                               threshold_rel=0.5, num_peaks=k)
    except ImportError:
        peaks = np.array([np.unravel_index(np.argmax(q), q.shape)])
    if len(peaks) < 2:
        return 0.0

    diag = float(np.hypot(*q.shape))
    ws = np.array([q[p[0], p[1]] for p in peaks], float)
    ws = ws / (ws.sum() + EPS)

    tot = wsum = 0.0
    for i in range(len(peaks)):
        for j in range(i + 1, len(peaks)):
            pi, pj = peaks[i], peaks[j]
            d_pos = np.hypot(pi[0] - pj[0], pi[1] - pj[1]) / diag
            d_ang = abs(ang[pi[0], pi[1]] - ang[pj[0], pj[1]]) % np.pi
            d_ang = min(d_ang, np.pi - d_ang) / (np.pi / 2)   # pi-periodic
            d_w = abs(w[pi[0], pi[1]] - w[pj[0], pj[1]]) / 150.0
            wt = ws[i] * ws[j]
            tot += wt * (0.5 * d_pos + 0.35 * d_ang + 0.15 * d_w)
            wsum += wt
    return float(tot / (wsum + EPS))


# --------------------------------------------------------------------------- #
# S3 — MC-dropout dispersion (T forward passes)
# --------------------------------------------------------------------------- #
def circular_std(angles: np.ndarray, period: float = np.pi) -> float:
    """Circular std for pi-periodic gripper angles.

    A plain np.std over angles near the 0/pi wrap reports a huge spread for
    samples that are physically almost identical, which would make S3 fire on
    exactly the scenes where the model is most consistent.
    """
    a = np.asarray(angles, float) * (2 * np.pi / period)
    r = np.hypot(np.mean(np.cos(a)), np.mean(np.sin(a)))
    r = min(max(r, EPS), 1.0)
    return float(np.sqrt(-2.0 * np.log(r)) * (period / (2 * np.pi)))


def score_mc_dropout(net, x, T: int = 20, lam_theta: float = 0.35,
                     lam_w: float = 0.15) -> float:
    n_layers = enable_mc_dropout(net)
    if n_layers == 0:
        raise RuntimeError(
            "No dropout layers active — S3 would be identically zero and would "
            "silently look like a perfectly confident model. Retrain with "
            "train.dropout > 0 in the config.")
    us, vs, ths, wsv = [], [], [], []
    with torch.no_grad():
        for _ in range(T):
            q, a, w = post_process(*net(x))
            g = decode_grasp(q, a, w)
            us.append(g["x"]); vs.append(g["y"])
            ths.append(g["theta"]); wsv.append(g["width"])
    net.eval()
    diag = float(np.hypot(*q.shape))
    return float(0.5 * (np.std(us) + np.std(vs)) / diag
                 + lam_theta * circular_std(np.asarray(ths))
                 + lam_w * np.std(wsv) / 150.0)


# --------------------------------------------------------------------------- #
# S4 — evidential (one forward pass)
# --------------------------------------------------------------------------- #
def score_evidential(ev_net, x):
    with torch.no_grad():
        out = ev_net(x)
    # select the pixel from the DETECTOR's quality map (pos), then read the
    # evidential parameters there. gamma is not the detector; pos is.
    q = out["pos"].squeeze().cpu().numpy()
    idx = np.unravel_index(np.argmax(q), q.shape)
    nu = float(out["nu"].squeeze().cpu().numpy()[idx])
    al = float(out["alpha"].squeeze().cpu().numpy()[idx])
    be = float(out["beta"].squeeze().cpu().numpy()[idx])
    return (float(be / max(al - 1.0, EPS)),               # aleatoric
            float(be / max(nu * (al - 1.0), EPS)))        # epistemic


# --------------------------------------------------------------------------- #
# Gate features
# --------------------------------------------------------------------------- #
def extract_feature(net, x) -> np.ndarray:
    """Pooled summary of the four heads — the input to the learned gate."""
    with torch.no_grad():
        outs = net(x)
    feats = []
    for m in outs:
        f = m.flatten(1)
        feats += [f.mean(1, keepdim=True), f.amax(1, keepdim=True),
                  f.std(1, keepdim=True)]
    return torch.cat(feats, 1).squeeze(0).cpu().numpy()


# --------------------------------------------------------------------------- #
def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--config", default="configs/cornell.yaml")
    ap.add_argument("--ckpt", required=True)
    ap.add_argument("--ev-ckpt", default=None)
    ap.add_argument("--split", default="test")
    ap.add_argument("--out", default="scores.npz")
    ap.add_argument("--device",
                    default="cuda" if torch.cuda.is_available() else "cpu")
    args = ap.parse_args()

    import yaml
    cfg = yaml.safe_load(open(args.config))
    size = cfg["data"]["out_size"]
    in_ch = 4 if cfg["data"].get("use_depth", True) else 3
    T = cfg["uq"].get("mc_samples", 20)
    drop = cfg["train"].get("dropout", 0.1)

    ds = make_dataset_factory(cfg)(args.split)
    print(f"[info] split='{args.split}'  N={len(ds)}  device={args.device}")

    net = load_backbone(args.ckpt, in_ch, args.device, drop, cfg=cfg)
    ev = load_evidential(args.ev_ckpt, in_ch, args.device, drop,
                         det_ckpt=args.ckpt, cfg=cfg) if args.ev_ckpt else None

    rec = {k: [] for k in ["idx", "obj_id", "success", "s_conf", "s_ent",
                           "s_mc", "s_topk", "s_ev_al", "s_ev_ep"]}
    feats = []

    for i in range(len(ds)):
        sample = ds[i]
        x = sample["image"][None].to(args.device)

        with torch.no_grad():
            raw = net(x)
            q, ang, w = post_process(*raw)
        g = decode_grasp(q, ang, w)

        rec["idx"].append(sample["idx"])
        rec["obj_id"].append(sample["obj_id"])
        rec["success"].append(grasp_success(g, sample["gt_grasps"]))
        rec["s_conf"].append(score_conf(q))
        rec["s_ent"].append(score_entropy(q))
        rec["s_topk"].append(score_topk_disagreement(q, ang, w))
        rec["s_mc"].append(score_mc_dropout(net, x, T=T))
        feats.append(extract_feature(net, x))
        if ev is not None:
            a, e = score_evidential(ev, x)
            rec["s_ev_al"].append(a)
            rec["s_ev_ep"].append(e)

        if (i + 1) % 100 == 0:
            print(f"  {i+1}/{len(ds)}")

    arrays = {k: np.asarray(v) for k, v in rec.items() if len(v)}
    arrays["feat"] = np.asarray(feats)
    assert len(np.unique(arrays["idx"])) == len(arrays["idx"]), \
        "duplicate indices — is the dataset order non-deterministic?"

    np.savez_compressed(args.out, **arrays)
    print(f"\n[ok] {len(arrays['success'])} samples -> {args.out}")
    print(f"     success rate = {arrays['success'].mean():.4f}")
    print(f"     scores cached: {[k for k in arrays if k.startswith('s_')]}")
    print("\nBACK THIS FILE UP NOW — it is the only expensive artefact here.")


if __name__ == "__main__":
    main()


In [ ]:
%%writefile selective_grasp.py
"""
selective_grasp.py
==================
Rejection rules and risk-coverage analysis for grasp prediction.

Input contract
--------------
This module deliberately knows nothing about GR-ConvNet. It consumes a cached
.npz produced by `extract_scores.py` containing, for N test samples:

    success : (N,) int/bool   1 if the predicted grasp satisfies the rectangle
                              metric (IoU > 0.25 and angle error < 30 deg)
    s_conf  : (N,) float      S1  1 - peak quality
    s_ent   : (N,) float      S2  spatial entropy of the quality map
    s_mc    : (N,) float      S3  MC-dropout pose dispersion      (optional)
    s_ev_ep : (N,) float      S4  evidential epistemic dispersion (optional)
    s_ev_al : (N,) float      S4  evidential aleatoric dispersion (optional)
    s_topk  : (N,) float      S5  top-k grasp disagreement

Convention: for every score, LARGER = LESS TRUSTWORTHY. Accept iff s <= tau.

Usage
-----
    python selective_grasp.py --scores scores.npz --alpha 0.05 --delta 0.10 \
                              --out results/

Outputs Table 1 / Table 2 / Table 3 of the paper as CSV + markdown, and the
risk-coverage figure.
"""

from __future__ import annotations

import argparse
import itertools
import json
import os
from dataclasses import dataclass, asdict

import numpy as np
from scipy.stats import binom

EPS = 1e-12

# numpy >= 2.0 renamed trapz -> trapezoid
_trapz = getattr(np, "trapezoid", None) or np.trapz


# --------------------------------------------------------------------------- #
# 1. Risk-coverage machinery
# --------------------------------------------------------------------------- #

def risk_coverage_curve(scores: np.ndarray, loss: np.ndarray):
    """Full risk-coverage curve, evaluated at every achievable coverage.

    Parameters
    ----------
    scores : (N,) larger = less trustworthy
    loss   : (N,) in [0,1]; here the 0/1 failure indicator

    Returns
    -------
    coverage : (N,) increasing from 1/N to 1
    risk     : (N,) selective risk at each coverage
    """
    order = np.argsort(scores, kind="mergesort")     # ascending uncertainty
    l_sorted = loss[order]
    n = len(loss)
    coverage = np.arange(1, n + 1) / n
    risk = np.cumsum(l_sorted) / np.arange(1, n + 1)
    return coverage, risk


def aurc(scores: np.ndarray, loss: np.ndarray) -> float:
    """Area under the risk-coverage curve (lower is better)."""
    _, risk = risk_coverage_curve(scores, loss)
    return float(np.mean(risk))


def oracle_aurc(loss: np.ndarray) -> float:
    """AURC of the hindsight-optimal gate (sort by true loss)."""
    return aurc(loss.astype(float), loss)


def excess_aurc(scores: np.ndarray, loss: np.ndarray) -> float:
    return aurc(scores, loss) - oracle_aurc(loss)


def risk_at_coverage(scores: np.ndarray, loss: np.ndarray, cov: float) -> float:
    """Selective risk when the gate accepts the `cov` fraction of lowest scores."""
    coverage, risk = risk_coverage_curve(scores, loss)
    idx = int(np.clip(np.searchsorted(coverage, cov), 0, len(risk) - 1))
    return float(risk[idx])


def coverage_at_risk(scores: np.ndarray, loss: np.ndarray, alpha: float,
                     n_min: int = 30) -> float:
    """Largest coverage whose empirical selective risk stays <= alpha.

    Returns 0.0 if no threshold with at least `n_min` accepted samples works.
    """
    coverage, risk = risk_coverage_curve(scores, loss)
    n = len(loss)
    valid = (risk <= alpha) & (np.arange(1, n + 1) >= n_min)
    return float(coverage[valid].max()) if valid.any() else 0.0


def ause(scores: np.ndarray, loss: np.ndarray) -> float:
    """Area under the sparsification error curve.

    IMPORTANT: with a *binary* loss the sparsification curve is exactly the
    reversed risk-coverage curve, and since sp_score >= sp_oracle pointwise,
    this quantity is algebraically identical to E-AURC. The column is therefore
    redundant here. To make the paper's ranking-vs-operating-point contrast
    real, compute AUSE on a *continuous* error (IoU deficit, angle error, or
    the regression residual used in the ICARCV benchmark) by passing that array
    as `loss` instead of the 0/1 indicator. Then AUSE and AURC genuinely
    measure different things and may disagree.
    """
    n = len(loss)
    order_s = np.argsort(-scores, kind="mergesort")   # remove most uncertain first
    order_o = np.argsort(-loss.astype(float), kind="mergesort")

    def sparsification(order):
        l = loss[order]
        # mean loss of the remaining set after removing the first k
        rem = np.cumsum(l[::-1])[::-1]
        counts = np.arange(n, 0, -1)
        return rem / counts

    sp_s = sparsification(order_s)
    sp_o = sparsification(order_o)
    return float(np.mean(np.abs(sp_s - sp_o)))


# --------------------------------------------------------------------------- #
# 2. Distribution-free risk control (Learn-then-Test)
# --------------------------------------------------------------------------- #

def _h1(a: float, b: float) -> float:
    """KL between two Bernoullis, used by the Hoeffding bound."""
    a = min(max(a, EPS), 1 - EPS)
    b = min(max(b, EPS), 1 - EPS)
    return a * np.log(a / b) + (1 - a) * np.log((1 - a) / (1 - b))


def hb_p_value(risk_hat: float, n: int, alpha: float) -> float:
    """Hoeffding-Bentkus p-value for H0: R > alpha, with bounded loss in [0,1].

    Small p means we can safely reject H0, i.e. the true risk is <= alpha.
    """
    if n <= 0:
        return 1.0
    if risk_hat >= alpha:
        return 1.0
    p_hoeff = float(np.exp(-n * _h1(risk_hat, alpha)))
    p_bent = float(np.e * binom.cdf(np.ceil(n * risk_hat), n, alpha))
    return min(1.0, min(p_hoeff, p_bent))


@dataclass
class CalibrationResult:
    tau: float | tuple
    p_value: float
    cal_risk: float
    cal_coverage: float
    n_accepted: int
    valid: bool          # False when no threshold could be certified


def ltt_calibrate_single(scores: np.ndarray, loss: np.ndarray,
                         alpha: float, delta: float,
                         n_grid: int = 200, n_min: int = 50) -> CalibrationResult:
    """Fixed-sequence Learn-then-Test for a single-threshold rejection rule.

    Walks candidate thresholds from most conservative (lowest coverage) to most
    liberal, rejecting H_j: R(tau_j) > alpha while p_j <= delta, and stops at
    the first failure. Guarantees P(R(tau_hat) <= alpha) >= 1 - delta.
    """
    qs = np.linspace(0.0, 1.0, n_grid + 1)[1:]           # increasing coverage
    grid = np.quantile(scores, qs)
    grid = np.unique(grid)                                # keeps ascending order

    best = CalibrationResult(np.nan, 1.0, np.nan, 0.0, 0, False)
    for tau in grid:
        acc = scores <= tau
        n_acc = int(acc.sum())
        if n_acc < n_min:
            continue                                      # too few to certify
        r_hat = float(loss[acc].mean())
        p = hb_p_value(r_hat, n_acc, alpha)
        if p <= delta:
            best = CalibrationResult(float(tau), p, r_hat,
                                     n_acc / len(scores), n_acc, True)
        else:
            break                                         # fixed-sequence stop
    return best


def ltt_calibrate_conjunctive(score_a: np.ndarray, score_b: np.ndarray,
                              loss: np.ndarray, alpha: float, delta: float,
                              n_grid: int = 20, n_min: int = 50,
                              path_frac: float = 0.5,
                              rng: np.random.Generator | None = None
                              ) -> CalibrationResult:
    """LTT for the conjunctive rule  accept iff (s_a <= t_a) and (s_b <= t_b).

    The threshold space is 2-D so no total order exists. We split the
    calibration set: the first half only *orders* the candidate pairs by their
    coverage (defining a fixed sequence), the second half runs the hypothesis
    tests. Ordering on independent data keeps the sequence pre-specified with
    respect to the testing data, which is what fixed-sequence testing requires.
    """
    rng = rng or np.random.default_rng(0)
    n = len(loss)
    perm = rng.permutation(n)
    n_path = int(path_frac * n)
    idx_path, idx_test = perm[:n_path], perm[n_path:]

    qs = np.linspace(0.0, 1.0, n_grid + 1)[1:]
    grid_a = np.unique(np.quantile(score_a, qs))
    grid_b = np.unique(np.quantile(score_b, qs))
    pairs = list(itertools.product(grid_a, grid_b))

    # order the sequence by coverage measured on the path half only
    cov_path = [
        float(((score_a[idx_path] <= ta) & (score_b[idx_path] <= tb)).mean())
        for ta, tb in pairs
    ]
    order = np.argsort(cov_path)                     # conservative -> liberal

    sa, sb, ls = score_a[idx_test], score_b[idx_test], loss[idx_test]
    best = CalibrationResult((np.nan, np.nan), 1.0, np.nan, 0.0, 0, False)
    for j in order:
        ta, tb = pairs[j]
        acc = (sa <= ta) & (sb <= tb)
        n_acc = int(acc.sum())
        if n_acc < n_min:
            continue
        r_hat = float(ls[acc].mean())
        p = hb_p_value(r_hat, n_acc, alpha)
        if p <= delta:
            best = CalibrationResult((float(ta), float(tb)), p, r_hat,
                                     n_acc / len(ls), n_acc, True)
        else:
            break
    return best


def naive_threshold(scores: np.ndarray, loss: np.ndarray, alpha: float,
                    n_min: int = 50) -> float:
    """The uncalibrated baseline: largest tau with empirical cal risk <= alpha.

    Included precisely to show it over-covers and violates alpha on fresh data.
    """
    order = np.argsort(scores, kind="mergesort")
    s_sorted, l_sorted = scores[order], loss[order]
    run = np.cumsum(l_sorted) / np.arange(1, len(loss) + 1)
    ok = (run <= alpha) & (np.arange(1, len(loss) + 1) >= n_min)
    return float(s_sorted[ok][-1]) if ok.any() else float(s_sorted[0])


# --------------------------------------------------------------------------- #
# 3. Logistic fusion (rule R3)
# --------------------------------------------------------------------------- #

def fit_logistic_fusion(S_cal: np.ndarray, loss_cal: np.ndarray,
                        n_iter: int = 500, lr: float = 0.5, l2: float = 1e-3):
    """Plain gradient-descent logistic regression predicting failure.

    Kept dependency-free and tiny on purpose; swap for sklearn if preferred.
    Returns (w, b, mu, sd) where mu/sd standardise the score matrix.
    """
    mu, sd = S_cal.mean(0), S_cal.std(0) + EPS
    X = (S_cal - mu) / sd
    y = loss_cal.astype(float)
    w = np.zeros(X.shape[1])
    b = 0.0
    for _ in range(n_iter):
        z = X @ w + b
        p = 1.0 / (1.0 + np.exp(-z))
        g = p - y
        w -= lr * (X.T @ g / len(y) + l2 * w)
        b -= lr * g.mean()
    return w, b, mu, sd


def apply_logistic_fusion(S: np.ndarray, params) -> np.ndarray:
    w, b, mu, sd = params
    z = ((S - mu) / sd) @ w + b
    return 1.0 / (1.0 + np.exp(-z))          # predicted failure prob = score


# --------------------------------------------------------------------------- #
# 4. Experiment drivers -> the three tables of the paper
# --------------------------------------------------------------------------- #

SCORE_KEYS = [
    ("s_conf",  "S1 negative peak quality"),
    ("s_ent",   "S2 quality-map entropy"),
    ("s_mc",    "S3 MC-dropout dispersion"),
    ("s_ev_ep", "S4 evidential (epistemic)"),
    ("s_ev_al", "S4 evidential (aleatoric)"),
    ("s_topk",  "S5 top-k disagreement"),
]


def load_scores(path: str):
    d = np.load(path)
    loss = 1.0 - d["success"].astype(float)
    scores = {k: d[k].astype(float) for k, _ in SCORE_KEYS if k in d}
    if not scores:
        raise ValueError("no recognised score arrays found in %s" % path)
    return scores, loss


def table1(scores: dict, loss: np.ndarray, alphas=(0.10, 0.05)):
    """Which score gives the best operating curve."""
    rows = []
    for key, name in SCORE_KEYS:
        if key not in scores:
            continue
        s = scores[key]
        row = {
            "score": name,
            "AURC": aurc(s, loss),
            "E-AURC": excess_aurc(s, loss),
            "AUSE": ause(s, loss),
            "R@90": risk_at_coverage(s, loss, 0.90),
            "R@70": risk_at_coverage(s, loss, 0.70),
        }
        for a in alphas:
            row[f"C*(a={a})"] = coverage_at_risk(s, loss, a)
        rows.append(row)
    rows.append({"score": "Oracle", "AURC": oracle_aurc(loss), "E-AURC": 0.0,
                 "AUSE": 0.0, "R@90": risk_at_coverage(loss, loss, 0.90),
                 "R@70": risk_at_coverage(loss, loss, 0.70),
                 **{f"C*(a={a})": coverage_at_risk(loss, loss, a) for a in alphas}})
    return rows


def table2(scores: dict, loss: np.ndarray, alpha: float = 0.05,
           rng: np.random.Generator | None = None):
    """Do combinations help: R1 vs conjunctive R2 vs fused R3."""
    rng = rng or np.random.default_rng(0)
    rows = []

    # R1: best single score by AURC
    singles = {k: aurc(v, loss) for k, v in scores.items()}
    best_key = min(singles, key=singles.get)
    rows.append({"rule": "R1 best single", "scores": best_key,
                 "AURC": singles[best_key],
                 f"C*(a={alpha})": coverage_at_risk(scores[best_key], loss, alpha)})

    # R2: conjunctions of s_conf with each dispersion-style score
    base = "s_conf"
    if base in scores:
        for other in ("s_mc", "s_topk", "s_ev_ep", "s_ent"):
            if other not in scores:
                continue
            cov, risk_c = _conjunctive_curve(scores[base], scores[other], loss)
            rows.append({
                "rule": "R2 conjunctive",
                "scores": f"{base} AND {other}",
                "AURC": float(_trapz(risk_c, cov) / max(cov[-1] - cov[0], EPS)),
                f"C*(a={alpha})": float(cov[risk_c <= alpha].max())
                if (risk_c <= alpha).any() else 0.0,
            })

    # R3: logistic fusion, fitted on half and evaluated on the other half
    keys = [k for k, _ in SCORE_KEYS if k in scores]
    S = np.stack([scores[k] for k in keys], axis=1)
    n = len(loss)
    perm = rng.permutation(n)
    a, b = perm[: n // 2], perm[n // 2:]
    params = fit_logistic_fusion(S[a], loss[a])
    s_fused = apply_logistic_fusion(S[b], params)
    rows.append({"rule": "R3 logistic fusion", "scores": "+".join(keys),
                 "AURC": aurc(s_fused, loss[b]),
                 f"C*(a={alpha})": coverage_at_risk(s_fused, loss[b], alpha)})
    return rows


def _conjunctive_curve(sa: np.ndarray, sb: np.ndarray, loss: np.ndarray,
                       n_grid: int = 25):
    """Pareto risk-coverage frontier of the conjunctive rule over a 2-D grid."""
    qs = np.linspace(0.02, 1.0, n_grid)
    ga, gb = np.quantile(sa, qs), np.quantile(sb, qs)
    pts = []
    for ta, tb in itertools.product(ga, gb):
        acc = (sa <= ta) & (sb <= tb)
        k = int(acc.sum())
        if k < 30:
            continue
        pts.append((k / len(loss), float(loss[acc].mean())))
    pts.sort()
    cov = np.array([p[0] for p in pts])
    rk = np.array([p[1] for p in pts])
    # lower envelope: best (lowest) risk achievable at each coverage or above
    rk = np.minimum.accumulate(rk[::-1])[::-1]
    return cov, rk


def table3(scores: dict, loss: np.ndarray, alpha: float, delta: float,
           n_splits: int = 100, seed: int = 0):
    """Does the calibrated threshold hold, and what does validity cost."""
    rng = np.random.default_rng(seed)
    n = len(loss)
    key = min(scores, key=lambda k: aurc(scores[k], loss))
    s = scores[key]

    rec = {"naive": {"risk": [], "cov": [], "viol": []},
           "ltt": {"risk": [], "cov": [], "viol": []}}

    for _ in range(n_splits):
        perm = rng.permutation(n)
        cal, ev = perm[: n // 2], perm[n // 2:]

        t_naive = naive_threshold(s[cal], loss[cal], alpha)
        res_ltt = ltt_calibrate_single(s[cal], loss[cal], alpha, delta)

        for name, tau in (("naive", t_naive),
                          ("ltt", res_ltt.tau if res_ltt.valid else -np.inf)):
            acc = s[ev] <= tau
            if acc.sum() == 0:
                rec[name]["risk"].append(0.0)
                rec[name]["cov"].append(0.0)
                rec[name]["viol"].append(0.0)
                continue
            r = float(loss[ev][acc].mean())
            rec[name]["risk"].append(r)
            rec[name]["cov"].append(float(acc.mean()))
            rec[name]["viol"].append(float(r > alpha))

    rows = []
    for name in ("naive", "ltt"):
        rows.append({
            "rule": f"{name} (score={key})",
            "alpha": alpha,
            "delta": delta if name == "ltt" else None,
            "mean_eval_risk": float(np.mean(rec[name]["risk"])),
            "violation_rate": float(np.mean(rec[name]["viol"])),
            "mean_coverage": float(np.mean(rec[name]["cov"])),
            "coverage_sd": float(np.std(rec[name]["cov"])),
        })
    rows.append({
        "rule": "coverage lost to validity",
        "alpha": alpha, "delta": delta,
        "mean_eval_risk": None, "violation_rate": None,
        "mean_coverage": rows[0]["mean_coverage"] - rows[1]["mean_coverage"],
        "coverage_sd": None,
    })
    return rows


# --------------------------------------------------------------------------- #
# 5. Reporting
# --------------------------------------------------------------------------- #

def to_markdown(rows: list[dict]) -> str:
    if not rows:
        return "(empty)\n"
    cols = list(rows[0].keys())
    def fmt(v):
        if v is None:
            return "-"
        if isinstance(v, float):
            return f"{v:.4f}"
        return str(v)
    out = ["| " + " | ".join(cols) + " |",
           "|" + "|".join(["---"] * len(cols)) + "|"]
    for r in rows:
        out.append("| " + " | ".join(fmt(r.get(c)) for c in cols) + " |")
    return "\n".join(out) + "\n"


def plot_curves(scores: dict, loss: np.ndarray, path: str):
    try:
        import matplotlib
        matplotlib.use("Agg")
        import matplotlib.pyplot as plt
    except ImportError:
        print("[warn] matplotlib unavailable, skipping figure")
        return
    fig, ax = plt.subplots(figsize=(5.2, 3.6))
    for key, name in SCORE_KEYS:
        if key not in scores:
            continue
        cov, rk = risk_coverage_curve(scores[key], loss)
        m = cov >= 0.05
        ax.plot(cov[m], rk[m], lw=1.4, label=name)
    cov, rk = risk_coverage_curve(loss, loss)
    m = cov >= 0.05
    ax.plot(cov[m], rk[m], "k--", lw=1.0, label="Oracle")
    ax.set_xlabel("Coverage")
    ax.set_ylabel("Selective risk (failure rate among executed)")
    ax.legend(fontsize=6.5, frameon=False)
    ax.grid(alpha=0.25, lw=0.4)
    fig.tight_layout()
    fig.savefig(path, dpi=300)
    print(f"[ok] figure -> {path}")


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--scores", required=True, help="path to scores .npz")
    ap.add_argument("--alpha", type=float, default=0.05)
    ap.add_argument("--delta", type=float, default=0.10)
    ap.add_argument("--splits", type=int, default=100)
    ap.add_argument("--out", default="results")
    args = ap.parse_args()

    os.makedirs(args.out, exist_ok=True)
    scores, loss = load_scores(args.scores)
    print(f"[info] N={len(loss)}  base failure rate={loss.mean():.4f}")

    t1 = table1(scores, loss)
    t2 = table2(scores, loss, alpha=args.alpha)
    t3 = table3(scores, loss, args.alpha, args.delta, n_splits=args.splits)
    t3 += table3(scores, loss, 0.10, args.delta, n_splits=args.splits)

    md = ("## Table 1 - score comparison\n\n" + to_markdown(t1) +
          "\n## Table 2 - rule comparison\n\n" + to_markdown(t2) +
          "\n## Table 3 - calibration validity\n\n" + to_markdown(t3))
    with open(os.path.join(args.out, "tables.md"), "w") as f:
        f.write(md)
    with open(os.path.join(args.out, "tables.json"), "w") as f:
        json.dump({"table1": t1, "table2": t2, "table3": t3}, f, indent=2)
    plot_curves(scores, loss, os.path.join(args.out, "risk_coverage.png"))
    print(md)


if __name__ == "__main__":
    main()


# --------------------------------------------------------------------------- #
# 6. Paired significance testing for the score/rule comparisons
# --------------------------------------------------------------------------- #
#
# Tables 1 and 2 are comparisons, and the paper's headline claim IS a comparison
# ("the score that wins on AUSE loses on C*"). A bare point estimate cannot
# support that claim. Every score is evaluated on the SAME samples, so the
# comparisons must be PAIRED: resample sample indices once per replicate and
# recompute both metrics on that same resample. Pairing removes the shared
# dataset noise and gives far tighter intervals than treating the two numbers as
# independent estimates.

_METRICS = {
    "AURC": lambda s, l, **kw: aurc(s, l),
    "E-AURC": lambda s, l, **kw: excess_aurc(s, l),
    "C*": lambda s, l, alpha=0.05, n_min=30: coverage_at_risk(s, l, alpha, n_min),
    "R@cov": lambda s, l, cov=0.8, **kw: risk_at_coverage(s, l, cov),
}


def paired_bootstrap_ci(score_a: np.ndarray, score_b: np.ndarray,
                        loss: np.ndarray, metric: str = "AURC",
                        n_boot: int = 2000, conf: float = 0.95,
                        seed: int = 0, **metric_kw):
    """Paired bootstrap CI for metric(a) - metric(b).

    Returns dict with the observed difference, the percentile interval, and
    whether the interval excludes zero. If it does not exclude zero, the two
    scores are not distinguishable at this sample size and the paper must say so
    rather than declaring a winner.
    """
    rng = np.random.default_rng(seed)
    fn = _METRICS[metric]
    n = len(loss)
    obs = fn(score_a, loss, **metric_kw) - fn(score_b, loss, **metric_kw)

    diffs = np.empty(n_boot)
    for b in range(n_boot):
        idx = rng.integers(0, n, n)                 # same idx for both -> paired
        l = loss[idx]
        if l.sum() == 0 or l.sum() == len(l):       # degenerate resample
            diffs[b] = np.nan
            continue
        diffs[b] = fn(score_a[idx], l, **metric_kw) - fn(score_b[idx], l, **metric_kw)

    diffs = diffs[~np.isnan(diffs)]
    lo, hi = np.percentile(diffs, [(1 - conf) / 2 * 100, (1 + conf) / 2 * 100])
    return dict(metric=metric, observed_diff=float(obs),
                ci_lo=float(lo), ci_hi=float(hi),
                significant=bool(lo > 0 or hi < 0), n_boot=len(diffs))


def paired_permutation_test(score_a: np.ndarray, score_b: np.ndarray,
                            loss: np.ndarray, metric: str = "AURC",
                            n_perm: int = 5000, seed: int = 0, **metric_kw):
    """Two-sided paired permutation test for H0: the two scores are equivalent.

    Under H0 the label 'which score' is exchangeable within each sample, so we
    swap the two scores independently per sample and rebuild the null
    distribution of the metric difference. Stricter than the bootstrap and the
    right test for the headline pair, since H0 here is genuine equivalence
    rather than a statement about the size of an effect.
    """
    rng = np.random.default_rng(seed)
    fn = _METRICS[metric]
    n = len(loss)
    obs = fn(score_a, loss, **metric_kw) - fn(score_b, loss, **metric_kw)

    null = np.empty(n_perm)
    for p in range(n_perm):
        swap = rng.random(n) < 0.5
        a = np.where(swap, score_b, score_a)
        b = np.where(swap, score_a, score_b)
        null[p] = fn(a, loss, **metric_kw) - fn(b, loss, **metric_kw)

    # +1 correction keeps the p-value valid (never reports exactly 0)
    p_val = (np.sum(np.abs(null) >= abs(obs)) + 1) / (n_perm + 1)
    return dict(metric=metric, observed_diff=float(obs), p_value=float(p_val),
                significant=bool(p_val < 0.05), n_perm=n_perm)


def random_aurc(loss: np.ndarray) -> float:
    """AURC of a gate that abstains at random: just the base error rate.

    A random score orders nothing, so its selective risk equals the overall
    failure rate at every coverage. This is the floor every real score must beat.
    """
    return float(loss.mean())


def normalized_aurc(scores: np.ndarray, loss: np.ndarray) -> float:
    """AURC rescaled to [0,1]: 0 = oracle, 1 = random. Lower is better.

    This is the number to report when someone asks "how good is this score?"
    Raw AURC depends on the base error rate, so it is not comparable across
    datasets or splits; the normalized version is. A score at 0.5 has closed
    half the gap between random guessing and the hindsight-optimal gate.
    """
    a = aurc(scores, loss)
    o = oracle_aurc(loss)
    r = random_aurc(loss)
    if r - o < 1e-9:
        return 0.0
    return float((a - o) / (r - o))


def baseline_comparison(scores: dict, loss: np.ndarray):
    """One table answering 'is this good?' against oracle and random anchors.

    Reports raw AURC, normalized AURC (0=oracle, 1=random), and the fraction of
    the oracle-to-random gap each score closes. The 'gap closed' column is the
    single most interpretable quality number: 60% means the score recovers 60%
    of the achievable improvement over guessing.
    """
    o = oracle_aurc(loss)
    r = random_aurc(loss)
    rows = []
    for key in list(scores) + ["__oracle__", "__random__"]:
        if key == "__oracle__":
            a, name = o, "ORACLE (upper bound)"
        elif key == "__random__":
            a, name = r, "RANDOM (lower bound)"
        else:
            a, name = aurc(scores[key], loss), key
        norm = 0.0 if (r - o) < 1e-9 else (a - o) / (r - o)
        rows.append({
            "score": name,
            "AURC": a,
            "norm_AURC": norm,               # 0 oracle, 1 random
            "gap_closed_%": 100.0 * (1.0 - norm),
        })
    rows.sort(key=lambda x: x["AURC"])
    return rows


def min_calibration_size(alpha, delta, target_coverage=0.5, n_grid=None):
    """Smallest calibration set that can certify risk <= alpha at confidence
    1 - delta via the Hoeffding bound, as a function of the budget.

    Learn-then-Test with a Hoeffding-Bentkus p-value rejects a threshold when
    the empirical risk r_hat on n calibration points satisfies, roughly,

        r_hat + sqrt( ln(1/delta) / (2 n) )  <=  alpha.

    In the best case r_hat -> 0 (a threshold that makes no mistakes on the
    calibration set), so certification needs the Hoeffding slack alone to fit
    under alpha:

        sqrt( ln(1/delta) / (2 n) )  <=  alpha
        =>  n  >=  ln(1/delta) / (2 alpha^2).

    This closed form is the paper's calibration-size result: it says the
    required calibration set grows as 1/alpha^2, which is exactly why a strict
    budget (alpha = 0.02) needs ~25x more data than a loose one (alpha = 0.10),
    and why Cornell (tens of calibration samples) cannot certify what Jacquard
    (thousands) can. Returns the minimum n (ceil).
    """
    import math
    if alpha <= 0:
        return float("inf")
    n_min = math.log(1.0 / delta) / (2.0 * alpha ** 2)
    return int(math.ceil(n_min))


def calibration_size_table(alphas, deltas):
    """The 1/alpha^2 growth as a table, for the paper's theory section."""
    rows = []
    for a in alphas:
        row = {"alpha": a}
        for d in deltas:
            row[f"n_min(delta={d})"] = min_calibration_size(a, d)
        rows.append(row)
    return rows


def conjunction_complementarity(score_a, score_b, loss):
    """Quantify WHY a conjunction of two scores helps: it helps exactly when the
    two scores catch DIFFERENT failures.

    Returns:
        corr            -- correlation between the two scores (low = they carry
                           independent information, the precondition for a useful
                           conjunction).
        unique_a        -- fraction of failures ranked risky by A but not B.
        unique_b        -- fraction of failures ranked risky by B but not A.
        joint           -- fraction caught by both.
        complementarity -- unique_a + unique_b: the failures a single score would
                           miss but the pair catches. High = the conjunction is
                           worth the second score.

    This is the analysis that turns "the conjunction had higher coverage" (an
    observation) into "the conjunction helps because these two scores fail on
    disjoint cases" (a mechanism) -- the difference between a number and a
    contribution.
    """
    a = np.asarray(score_a); b = np.asarray(score_b)
    fail = np.asarray(loss).astype(bool)
    corr = float(np.corrcoef(a, b)[0, 1])

    # "flagged risky" = above-median score, among the actual failures
    ma, mb = np.median(a), np.median(b)
    fa, fb = (a > ma), (b > mb)
    F = fail
    n_fail = int(F.sum())
    if n_fail == 0:
        return dict(corr=corr, unique_a=0.0, unique_b=0.0, joint=0.0,
                    complementarity=0.0)
    only_a = int((F & fa & ~fb).sum()) / n_fail
    only_b = int((F & ~fa & fb).sum()) / n_fail
    both = int((F & fa & fb).sum()) / n_fail
    return dict(corr=corr, unique_a=only_a, unique_b=only_b, joint=both,
                complementarity=only_a + only_b)


def best_conjunction(scores, loss, alpha, delta, base_key=None):
    """Search score pairs for the conjunction with the highest certified
    coverage at (alpha, delta). Returns the winning pair and its coverage,
    alongside the best single score, so the gain from conjoining is explicit.
    """
    keys = list(scores)
    single = {k: coverage_at_risk(scores[k], loss, alpha) for k in keys}
    best_single = max(single, key=single.get)
    best_pair, best_cov = None, -1.0
    for i, ka in enumerate(keys):
        for kb in keys[i + 1:]:
            res = ltt_calibrate_conjunctive(scores[ka], scores[kb],
                                            loss, alpha, delta)
            cov = res.get("coverage", 0.0) if isinstance(res, dict) else 0.0
            if cov > best_cov:
                best_cov, best_pair = cov, (ka, kb)
    return dict(best_pair=best_pair, pair_coverage=best_cov,
                best_single=best_single,
                single_coverage=single[best_single],
                gain=best_cov - single[best_single])


def significance_table(scores: dict, loss: np.ndarray, reference: str | None = None,
                       metric: str = "AURC", n_boot: int = 2000,
                       seed: int = 0, **metric_kw):
    """Compare every score against a reference (default: the cheapest, s_conf).

    s_conf is the right reference because it costs one forward pass. Any score
    requiring extra compute must beat it by a statistically distinguishable
    margin to justify itself; 'numerically better' is not a justification.
    """
    reference = reference or ("s_conf" if "s_conf" in scores else list(scores)[0])
    rows = []
    for key in scores:
        if key == reference:
            continue
        ci = paired_bootstrap_ci(scores[key], scores[reference], loss,
                                 metric=metric, n_boot=n_boot, seed=seed, **metric_kw)
        rows.append({
            "score": key,
            "vs": reference,
            f"delta_{metric}": ci["observed_diff"],
            "ci95_lo": ci["ci_lo"],
            "ci95_hi": ci["ci_hi"],
            "distinguishable": ci["significant"],
        })
    return rows


In [ ]:
%%writefile object_eval.py
"""
object_eval.py
==============
Per-object simulation and evaluation for the selective-grasp paper. Everything
here answers the paper's question -- "when NOT to grasp?" -- NOT GR-ConvNet's
question ("is the grasp accurate?"). The distinction drives every function:

  * GR-ConvNet reports success rate. We report, per object, whether the
    abstention rule EXECUTES or REJECTS, and whether that decision was right.
  * GR-ConvNet's physical trial asks "did the grasp hold?". Our lightweight
    trial asks "if we obey the rule, is the failure rate among EXECUTED grasps
    within budget?" -- an empirical check of the LTT guarantee, per object.

Three deliverables, matching the three requests:

  1. per_object_visual_records()  -> data for image + predicted grasp +
     accept/reject label + uncertainty, one record per object.
  2. simulate_execution()          -> a physics-free "execution" trial on the
     Jacquard test set: apply tau, measure realised risk on executed grasps,
     confirm it sits under alpha. This is the LTT guarantee made concrete.
  3. per_object_table()            -> success / rejection / executed-risk broken
     down by object, exposing which objects the rule protects against.

No physics engine. The Jacquard success label already encodes whether a
predicted grasp matches the annotated grasp set under the rectangle metric, so
"executing" a grasp on the test set means reading that label -- deterministic,
reproducible, and free of a PyBullet dependency that would not run cleanly on
Colab. Where a physical simulator would add value (contact dynamics, slip) is
noted in the docstrings so the paper can scope it as future work honestly.
"""

from __future__ import annotations

import numpy as np

EPS = 1e-12


# =========================================================================== #
# 1. Per-object visual records
# =========================================================================== #
def per_object_visual_records(idx, obj_id, success, score, tau,
                              max_per_object: int = 1):
    """One record per object for the qualitative figure.

    Returns a list of dicts with the fields a plotting routine needs:
        obj_id, idx (dataset index, to fetch the image), executed (bool),
        correct (bool), score, and outcome in {accepted_correct,
        accepted_wrong, rejected_would_fail, rejected_would_succeed}.

    The four outcomes are the whole point. GR-ConvNet figures show correct vs
    incorrect grasps; ours must show the DECISION crossed with the truth,
    because "rejected a grasp that would have failed" is the behaviour the paper
    claims and "accepted one that failed" is the residual risk it must own.
    """
    idx = np.asarray(idx)
    obj_id = np.asarray(obj_id)
    success = np.asarray(success).astype(int)
    score = np.asarray(score)

    executed = score <= tau
    records = []
    for oid in np.unique(obj_id):
        mask = obj_id == oid
        # per-object statistics, attached to every record for this object so a
        # panel can show "this object's grasps succeed X% of the time" and
        # "the rule executes Y% of them"
        obj_success_rate = float(success[mask].mean())
        obj_coverage = float(executed[mask].mean())
        n_obj = int(mask.sum())
        # prefer to showcase a rejected example if one exists for this object,
        # since rejections are the paper's contribution and the more informative
        # cell to display
        order = np.argsort(~(~executed[mask]))  # rejected first
        chosen = np.where(mask)[0][order][:max_per_object]
        for j in chosen:
            ex = bool(executed[j])
            ok = bool(success[j])
            if ex and ok:
                outcome = "accepted_correct"
            elif ex and not ok:
                outcome = "accepted_wrong"          # residual risk
            elif not ex and not ok:
                outcome = "rejected_would_fail"      # correct rejection
            else:
                outcome = "rejected_would_succeed"   # cost of caution
            records.append(dict(obj_id=int(oid), idx=int(idx[j]),
                                executed=ex, correct=ok,
                                score=float(score[j]), outcome=outcome,
                                obj_success_rate=obj_success_rate,
                                obj_coverage=obj_coverage, n_obj=n_obj))
    return records


def outcome_counts(records):
    """Tally the four decision outcomes across a set of records."""
    counts = {"accepted_correct": 0, "accepted_wrong": 0,
              "rejected_would_fail": 0, "rejected_would_succeed": 0}
    for r in records:
        counts[r["outcome"]] += 1
    return counts


# =========================================================================== #
# 2. Physics-free execution simulation (the LTT guarantee, made concrete)
# =========================================================================== #
def simulate_execution(success, score, tau, alpha=None):
    """'Execute' every grasp the rule accepts; measure realised risk.

    On the Jacquard test set a grasp's success label already says whether the
    predicted grasp matches the annotation under the rectangle metric, so
    executing the accepted subset means reading those labels. Returns:

        n_total, n_executed, coverage,
        executed_risk   -- failure rate AMONG executed grasps (the quantity the
                           budget constrains),
        rejected_risk   -- failure rate among rejected grasps (should be HIGHER;
                           if it isn't, the score isn't ordering by risk),
        within_budget   -- executed_risk <= alpha (None if alpha not given).

    A physical simulator would additionally model contact and slip, so the true
    execution risk could exceed this metric-based estimate; that gap is the
    honest scope of "future work: closed-loop physical trials".
    """
    success = np.asarray(success).astype(int)
    fail = 1 - success
    score = np.asarray(score)

    executed = score <= tau
    n_exec = int(executed.sum())
    n_rej = int((~executed).sum())

    exec_risk = float(fail[executed].mean()) if n_exec else 0.0
    rej_risk = float(fail[~executed].mean()) if n_rej else 0.0

    out = dict(
        n_total=len(success),
        n_executed=n_exec,
        n_rejected=n_rej,
        coverage=n_exec / max(len(success), 1),
        executed_risk=exec_risk,
        rejected_risk=rej_risk,
        risk_reduction=float(fail.mean()) - exec_risk,   # vs executing all
    )
    if alpha is not None:
        out["alpha"] = alpha
        out["within_budget"] = exec_risk <= alpha + 1e-9
    return out


def simulate_execution_curve(success, score, taus):
    """Run simulate_execution across a sweep of thresholds, for a plot."""
    return [dict(tau=float(t), **simulate_execution(success, score, t))
            for t in taus]


# =========================================================================== #
# 3. Per-object breakdown
# =========================================================================== #
def per_object_table(obj_id, success, score, tau, alpha=None,
                     min_samples: int = 1, sort_by="executed_risk"):
    """Success / rejection / executed-risk per object.

    Exposes which objects the detector struggles with and whether the rule
    rejects them. An object with low raw success but high rejection rate is a
    WIN for the paper: the rule recognised a hard object and abstained. An
    object with low success AND low rejection is the residual-risk case worth
    showing.
    """
    obj_id = np.asarray(obj_id)
    success = np.asarray(success).astype(int)
    fail = 1 - success
    score = np.asarray(score)
    executed = score <= tau

    rows = []
    for oid in np.unique(obj_id):
        m = obj_id == oid
        n = int(m.sum())
        if n < min_samples:
            continue
        n_exec = int(executed[m].sum())
        exec_fail = fail[m & executed]
        rows.append({
            "obj_id": int(oid),
            "n": n,
            "raw_success": float(success[m].mean()),
            "rejection_rate": float((~executed[m]).mean()),
            "coverage": n_exec / max(n, 1),
            "executed_risk": float(exec_fail.mean()) if len(exec_fail) else 0.0,
        })
    reverse = sort_by in ("executed_risk", "rejection_rate")
    rows.sort(key=lambda r: r.get(sort_by, 0.0), reverse=reverse)
    return rows


def object_summary(rows, alpha=None):
    """Aggregate the per-object table into a few headline numbers."""
    if not rows:
        return {}
    ex_risks = np.array([r["executed_risk"] for r in rows])
    cov = np.array([r["coverage"] for r in rows])
    summary = dict(
        n_objects=len(rows),
        mean_coverage=float(cov.mean()),
        mean_executed_risk=float(ex_risks.mean()),
        worst_executed_risk=float(ex_risks.max()),
        objects_fully_rejected=int((cov == 0).sum()),
    )
    if alpha is not None:
        summary["objects_over_budget"] = int((ex_risks > alpha + 1e-9).sum())
        summary["frac_objects_within_budget"] = \
            float((ex_risks <= alpha + 1e-9).mean())
    return summary


def alpha_sweep(score, loss, obj_id, alphas, delta=0.10,
                ltt_calibrate=None, simulate=None, per_object=None):
    """Compare candidate risk budgets side by side so alpha can be chosen with
    evidence, not by habit.

    For each alpha it calibrates a threshold by LTT, simulates execution, and
    counts how many objects the rule actually discriminates on (partial
    coverage: neither fully executed nor fully rejected). The columns that
    matter when choosing:

        ltt_valid            - was a threshold certifiable at this alpha and
                               this calibration size? FALSE means the budget is
                               too strict for the data you have.
        coverage             - fraction of grasps executed.
        executed_risk        - realised failure rate among executed grasps.
        objs_discriminating  - objects where the rule made a non-trivial choice.
        trivial              - base failure rate <= alpha, so "execute all"
                               already meets the budget and the rule is idle.

    The sweet spot is the strictest alpha that is still ltt_valid AND not
    trivial: there the guarantee holds and the rejection rule is doing real
    work. Pass the selective_grasp functions in (kept as arguments so this
    module has no hard dependency on them).
    """
    import importlib
    if ltt_calibrate is None or simulate is None or per_object is None:
        sg = importlib.import_module("selective_grasp")
        ltt_calibrate = ltt_calibrate or sg.ltt_calibrate_single
        simulate = simulate or simulate_execution
        per_object = per_object or per_object_table

    loss = np.asarray(loss).astype(float)
    base = float(loss.mean())
    rows = []
    for a in alphas:
        res = ltt_calibrate(score, loss, a, delta)
        tau = res.tau if res.valid else float("-inf")
        sim = simulate(1 - loss, score, tau, alpha=a)
        tbl = per_object(obj_id, 1 - loss, score, tau, alpha=a)
        discriminating = sum(1 for r in tbl if 0.0 < r["coverage"] < 1.0)
        rows.append(dict(
            alpha=a,
            ltt_valid=bool(res.valid),
            coverage=round(sim["coverage"], 3),
            executed_risk=round(sim["executed_risk"], 4),
            within_budget=sim.get("within_budget"),
            objs_discriminating=f"{discriminating}/{len(tbl)}",
            trivial=(base <= a + 1e-9),
        ))
    return rows


def grid_search(scores, loss, obj_id, alphas, delta=0.10,
                ltt_calibrate=None):
    """Full (score x alpha) grid for choosing the operating point.

    For every uncertainty score and every candidate budget, report whether LTT
    certifies a threshold, the resulting coverage and executed risk, whether the
    budget is met, and whether it is trivial (base failure rate <= alpha). This
    is the table to scan when picking WHICH score to threshold and at WHAT
    budget -- the two decisions the paper's operating point rests on.

    Unlike alpha_sweep (one score, many alphas), this crosses every score with
    every alpha, so the winning combination is visible in one place.
    """
    import importlib
    if ltt_calibrate is None:
        ltt_calibrate = importlib.import_module("selective_grasp").ltt_calibrate_single

    loss = np.asarray(loss).astype(float)
    base = float(loss.mean())
    grid = []
    for name, sc in scores.items():
        sc = np.asarray(sc)
        for a in alphas:
            res = ltt_calibrate(sc, loss, a, delta)
            tau = res.tau if res.valid else float("-inf")
            sim = simulate_execution(1 - loss, sc, tau, alpha=a)
            grid.append(dict(
                score=name, alpha=a, ltt_valid=bool(res.valid),
                coverage=round(sim["coverage"], 3),
                exec_risk=round(sim["executed_risk"], 4),
                within_budget=sim.get("within_budget"),
                trivial=(base <= a + 1e-9),
            ))
    return grid


def best_operating_point(grid):
    """Highest-coverage (score, alpha) that is certified, within budget, and
    non-trivial. Returns the row, or None with a reason if none qualifies.

    The ranking is deliberate: among rows where the guarantee actually holds and
    the rule is doing real work, more coverage is strictly better -- it means
    executing more grasps at the same risk ceiling. Trivial rows are excluded
    because their coverage is high for the wrong reason (the budget was slack).
    """
    usable = [r for r in grid
              if r["ltt_valid"] and r["within_budget"] and not r["trivial"]]
    if usable:
        best = max(usable, key=lambda r: r["coverage"])
        return best, "highest coverage among certified, within-budget, non-trivial"
    certified = [r for r in grid if r["ltt_valid"] and not r["trivial"]]
    if certified:
        best = max(certified, key=lambda r: r["coverage"])
        return best, ("no row meets budget exactly; showing best certified "
                      "non-trivial (report the small overshoot honestly)")
    return None, ("nothing certifies non-trivially at this calibration size; "
                  "more calibration data is needed -- this is what Jacquard "
                  "buys over Cornell")


def recommend_alpha(sweep_rows):
    """Pick the strictest alpha that is both certifiable and non-trivial.

    Returns (alpha, reason). If none qualifies, falls back to the strictest
    certifiable alpha and says so, or to the loosest alpha if nothing certifies.
    """
    usable = [r for r in sweep_rows if r["ltt_valid"] and not r["trivial"]]
    if usable:
        best = min(usable, key=lambda r: r["alpha"])
        return best["alpha"], "strictest budget that is certifiable and non-trivial"
    certifiable = [r for r in sweep_rows if r["ltt_valid"]]
    if certifiable:
        best = min(certifiable, key=lambda r: r["alpha"])
        return best["alpha"], ("no non-trivial budget certifies at this "
                               "calibration size; using strictest certifiable")
    loosest = max(sweep_rows, key=lambda r: r["alpha"])
    return loosest["alpha"], ("NOTHING certifies at this calibration size; "
                              "report this as a limitation and use more data")


def regime_check(success, alpha) -> dict:
    """Flag whether the detector leaves enough failures for the study to be
    informative at this budget.

    When the base failure rate is at or below alpha, the trivial rule "execute
    everything" already satisfies the budget, so the rejection rule has nothing
    to prove and every per-object executed-risk collapses to ~0. This is not a
    bug; it is a statement about the operating point. On Jacquard, where the
    detector reaches ~93%, the interesting budgets are the STRICT ones (well
    below the ~7% base failure rate). The function returns the base rate, whether
    the budget is non-trivial, and a suggested alpha if it is not.
    """
    success = np.asarray(success).astype(int)
    base_fail = float((1 - success).mean())
    non_trivial = base_fail > alpha + 1e-9
    out = dict(base_failure_rate=base_fail, alpha=alpha,
               budget_non_trivial=non_trivial)
    if not non_trivial:
        # suggest a budget that actually forces the rule to discriminate:
        # roughly half the base failure rate, so "execute all" is NOT a solution
        out["suggested_alpha"] = round(max(base_fail / 2.0, 0.005), 3)
        out["note"] = (
            "Base failure rate <= alpha: 'execute all' already meets the "
            "budget, so the rejection rule is not exercised and per-object "
            "risks read ~0. Use a stricter alpha (suggested) to make the "
            "selective-prediction contribution visible.")
    return out


def rejection_breakdown(success, score, tau):
    """Diagnose over- vs under-rejection with one call.

    Splits the rejected pile into grasps that WOULD have succeeded (the cost of
    caution -- over-rejection) and grasps that would have failed (correct
    rejections), and the executed pile into correct and wrong. Returns counts,
    the four rates, and two headline numbers:

        over_rejection_rate  -- of everything rejected, the fraction that would
                                have succeeded. High means the threshold is too
                                strict: it is throwing away good grasps.
        precision_of_rejection -- of everything rejected, the fraction that
                                would indeed have failed. This is the number to
                                report: "when the rule says no, it is right X%
                                of the time."
    """
    success = np.asarray(success).astype(int)
    fail = 1 - success
    executed = np.asarray(score) <= tau
    rejected = ~executed

    n_rej = int(rejected.sum())
    n_exec = int(executed.sum())
    rej_would_fail = int(fail[rejected].sum())
    rej_would_succeed = int(success[rejected].sum())
    exec_correct = int(success[executed].sum())
    exec_wrong = int(fail[executed].sum())

    return dict(
        n_total=len(success),
        n_executed=n_exec,
        n_rejected=n_rej,
        rejected_would_fail=rej_would_fail,
        rejected_would_succeed=rej_would_succeed,
        executed_correct=exec_correct,
        executed_wrong=exec_wrong,
        over_rejection_rate=(rej_would_succeed / n_rej) if n_rej else 0.0,
        precision_of_rejection=(rej_would_fail / n_rej) if n_rej else 0.0,
        coverage=n_exec / max(len(success), 1),
    )


def rejection_summary_line(success, score, tau) -> str:
    """A one-line human-readable summary to print above the figure."""
    b = rejection_breakdown(success, score, tau)
    if b["n_rejected"] == 0:
        return "Nothing rejected at this threshold."
    return (
        f"Rejected {b['n_rejected']}/{b['n_total']} grasps: "
        f"{b['precision_of_rejection']*100:.0f}% would truly have failed "
        f"(correct rejections), "
        f"{b['over_rejection_rate']*100:.0f}% would have succeeded "
        f"(over-rejection, the cost of caution). "
        f"Executed {b['n_executed']}: "
        f"{b['executed_correct']} correct, {b['executed_wrong']} wrong."
    )


# =========================================================================== #
# 4. Rendering the per-object figure (the piece that draws actual images)
# =========================================================================== #
def render_record(dataset, model, idx, device="cpu", width_scale=100.0):
    """Fetch image + predicted grasp + annotations for ONE dataset index.

    Returns the dict make_figures.draw_panel expects: image, quality, pred, gts.
    This is the bridge the earlier notebook left as a sketch -- it actually runs
    the detector on the stored image so the panel shows a real prediction, not a
    placeholder.
    """
    import torch
    from models_grconvnet import post_process
    from train import decode_grasp
    from metrics_grasp import corners_to_params

    sample = dataset[idx]
    x = sample["image"]
    x = x[None] if hasattr(x, "ndim") else x
    if not hasattr(x, "to"):
        x = torch.as_tensor(np.asarray(x))[None]
    with torch.no_grad():
        out = model(x.to(device))
    if isinstance(out, dict):                       # evidential wrapper
        pos, cos, sin, wd = out["pos"], out["cos"], out["sin"], out["width"]
    else:
        pos, cos, sin, wd = out
    q, ang, w = post_process(pos, cos, sin, wd)
    g = decode_grasp(q, ang, w)

    # RGB for display: first three channels, H,W,C
    img = np.asarray(sample["image"])
    if img.shape[0] in (3, 4):
        img = np.transpose(img[:3], (1, 2, 0))
    img = np.clip(img, 0, 1)

    gts = []
    for rect in np.asarray(sample["gt_grasps"]):
        cx, cy, th, ww, hh = corners_to_params(rect)
        gts.append(dict(x=float(cx), y=float(cy), theta=float(th),
                        width=float(min(ww, hh))))
    return dict(image=img, quality=q, pred=g, gts=gts, alts=[])


def figure_per_object(dataset, model, records, out_path, device="cpu",
                      n_cols=4, max_panels=12, prefer=("rejected_would_fail",
                                                       "accepted_wrong")):
    """Grid of per-object panels, foregrounding the decision-relevant outcomes.

    `records` come from per_object_visual_records(). Panels are ordered so the
    outcomes that carry the paper -- correct rejections and residual-risk
    acceptances -- appear first, because those are what a reviewer inspects. A
    green border means executed, red means rejected; the caption states the
    outcome. Falls back to the synthetic renderer only if the dataset/model
    can't produce an image, so the layout is always viewable.
    """
    import matplotlib
    matplotlib.use("Agg")
    import matplotlib.pyplot as plt
    from make_figures import draw_panel

    # order: preferred outcomes first, then the rest
    pref = [r for r in records if r["outcome"] in prefer]
    rest = [r for r in records if r["outcome"] not in prefer]
    chosen = (pref + rest)[:max_panels]
    if not chosen:
        raise ValueError("no records to render")

    n = len(chosen)
    n_cols = min(n_cols, n)
    n_rows = (n + n_cols - 1) // n_cols
    fig, axes = plt.subplots(n_rows, n_cols,
                             figsize=(2.4 * n_cols, 2.4 * n_rows))
    axes = np.atleast_1d(axes).ravel()

    label = {"rejected_would_fail": "Rejected — would fail  (correct)",
             "accepted_wrong": "Accepted — WRONG  (residual risk)",
             "accepted_correct": "Accepted — correct",
             "rejected_would_succeed": "Rejected — would succeed  (cost)"}

    for ax, r in zip(axes, chosen):
        try:
            s = render_record(dataset, model, r["idx"], device=device)
        except Exception:
            from make_figures import render_sample_demo
            s = render_sample_demo(r["idx"])
        # two-line caption: the decision outcome, then this object's grasp
        # success rate and how much of it the rule executes. The success rate is
        # the number requested; showing it next to the decision makes the
        # over/under-rejection visible at a glance.
        sr = r.get("obj_success_rate")
        cov = r.get("obj_coverage")
        stat = ""
        if sr is not None:
            stat = f"\nsuccess {sr*100:.0f}%"
            if cov is not None:
                stat += f" · executed {cov*100:.0f}%"
        draw_panel(ax, s, accepted=r["executed"], success=r["correct"],
                   subtitle=f"obj {r['obj_id']} · {label[r['outcome']]}{stat}")
    for ax in axes[n:]:
        ax.axis("off")

    fig.tight_layout()
    fig.savefig(out_path, dpi=150, bbox_inches="tight")
    plt.close(fig)
    return out_path


def cross_dataset_comparison(datasets, alphas, delta=0.10, ltt_calibrate=None):
    """Compare over-rejection and coverage across datasets at several budgets.

    `datasets` maps a name to (success_array, best_score_array). For each dataset
    and alpha it reports coverage, executed risk, and over-rejection, so the
    paper can show WHERE each dataset's operating regime is: a weaker detector
    (Cornell) does real work at moderate alpha, a stronger one (Jacquard) needs
    a strict alpha before the rejection rule is exercised at all.

    This is the table behind the "two datasets, two complementary regimes"
    argument -- the strongest framing available for using both.
    """
    import importlib
    if ltt_calibrate is None:
        ltt_calibrate = importlib.import_module("selective_grasp").ltt_calibrate_single

    rows = []
    for name, (success, score) in datasets.items():
        success = np.asarray(success).astype(int)
        loss = 1 - success
        base = float(success.mean())
        for a in alphas:
            res = ltt_calibrate(np.asarray(score), loss, a, delta)
            tau = res.tau if res.valid else float("-inf")
            b = rejection_breakdown(success, score, tau)
            rows.append(dict(
                dataset=name, base_success=round(base, 3), alpha=a,
                ltt_valid=bool(res.valid),
                coverage=round(b["coverage"], 3),
                over_rejection=round(b["over_rejection_rate"], 3),
                rej_precision=round(b["precision_of_rejection"], 3),
                trivial=((1 - base) <= a + 1e-9),
            ))
    return rows


# =========================================================================== #
# Self-test
# =========================================================================== #
def _selftest():
    def ok(c, m):
        print(f"  {'PASS' if c else 'FAIL'}  {m}")
        if not c:
            raise SystemExit(1)

    rng = np.random.default_rng(0)
    n = 400
    # scores correlate with failure: higher score -> more likely to fail
    u = rng.random(n)
    fail = (rng.random(n) < u).astype(int)
    success = 1 - fail
    score = u + 0.15 * rng.normal(size=n)
    obj_id = rng.integers(0, 20, n)          # 20 objects

    print("[execution simulation]")
    tau = np.quantile(score, 0.6)            # accept the 60% lowest-risk
    sim = simulate_execution(success, score, tau, alpha=0.15)
    ok(0 < sim["coverage"] < 1, "coverage strictly between 0 and 1")
    ok(sim["executed_risk"] < sim["rejected_risk"],
       "executed risk < rejected risk (score orders by risk)")
    ok(sim["risk_reduction"] > 0, "abstention reduces risk vs executing all")
    ok(sim["n_executed"] + sim["n_rejected"] == sim["n_total"],
       "executed + rejected == total")

    print("[monotone coverage sweep]")
    taus = np.quantile(score, np.linspace(0.1, 0.9, 9))
    curve = simulate_execution_curve(success, score, taus)
    covs = [c["coverage"] for c in curve]
    ok(all(covs[i] <= covs[i+1] + 1e-9 for i in range(len(covs)-1)),
       "coverage increases as tau rises")
    risks = [c["executed_risk"] for c in curve]
    ok(risks[0] <= risks[-1] + 1e-9,
       "executed risk grows as more (riskier) grasps are admitted")

    print("[per-object visual records]")
    recs = per_object_visual_records(np.arange(n), obj_id, success, score, tau)
    ok(len(recs) == len(np.unique(obj_id)), "one record per object by default")
    outs = {r["outcome"] for r in recs}
    ok(outs <= {"accepted_correct", "accepted_wrong",
                "rejected_would_fail", "rejected_would_succeed"},
       "every record has a valid outcome label")
    counts = outcome_counts(recs)
    ok(sum(counts.values()) == len(recs), "outcome tally covers all records")

    print("[per-object table]")
    tbl = per_object_table(obj_id, success, score, tau, alpha=0.15)
    ok(len(tbl) == len(np.unique(obj_id)), "one row per object")
    for r in tbl:
        ok(0 <= r["executed_risk"] <= 1, f"obj {r['obj_id']} risk in [0,1]")
        break
    ok(all(tbl[i]["executed_risk"] >= tbl[i+1]["executed_risk"]
           for i in range(len(tbl)-1)), "table sorted by executed risk desc")

    summ = object_summary(tbl, alpha=0.15)
    ok(summ["n_objects"] == len(tbl), "summary counts all objects")
    ok(0 <= summ["frac_objects_within_budget"] <= 1,
       "fraction within budget in [0,1]")

    print("[edge cases]")
    # tau below every score -> reject everything
    s0 = simulate_execution(success, score, score.min() - 1)
    ok(s0["n_executed"] == 0 and s0["coverage"] == 0.0, "reject-all handled")
    # tau above every score -> execute everything, risk == base rate
    s1 = simulate_execution(success, score, score.max() + 1)
    ok(s1["coverage"] == 1.0 and abs(s1["executed_risk"] - fail.mean()) < 1e-9,
       "execute-all risk equals base failure rate")

    print("[regime check]")
    all_good = np.ones(100, int)
    mostly_good = np.concatenate([np.ones(93, int), np.zeros(7, int)])
    r1 = regime_check(mostly_good, alpha=0.10)
    ok(not r1["budget_non_trivial"], "7% fail vs alpha=0.10 -> trivial budget flagged")
    ok("suggested_alpha" in r1 and r1["suggested_alpha"] < 0.07,
       "a stricter alpha is suggested")
    r2 = regime_check(mostly_good, alpha=0.03)
    ok(r2["budget_non_trivial"], "7% fail vs alpha=0.03 -> non-trivial, no flag")

    print("[alpha sweep]")
    sweep = alpha_sweep(score, fail.astype(float), obj_id,
                        [0.20, 0.15, 0.10, 0.05])
    ok(len(sweep) == 4, "one row per candidate alpha")
    ok(all("ltt_valid" in r and "coverage" in r for r in sweep),
       "sweep rows carry the decision columns")
    # coverage must be monotone non-increasing as alpha tightens
    covs = [r["coverage"] for r in sweep]
    ok(all(covs[i] >= covs[i+1] - 1e-9 for i in range(len(covs)-1)),
       "coverage does not increase as alpha tightens")
    a_rec, reason = recommend_alpha(sweep)
    ok(a_rec in [r["alpha"] for r in sweep], "recommended alpha is one of the candidates")
    ok(isinstance(reason, str) and len(reason) > 0, "recommendation gives a reason")

    print("[grid search]")
    sc_multi = {"a": score, "b": score + 0.1 * rng.normal(size=n),
                "c": score - 0.05 * rng.normal(size=n)}
    grid = grid_search(sc_multi, fail.astype(float), obj_id, [0.20, 0.15, 0.10])
    ok(len(grid) == 3 * 3, "grid has one row per (score, alpha) pair")
    ok(all(set(r) >= {"score","alpha","ltt_valid","coverage","exec_risk",
                      "within_budget","trivial"} for r in grid),
       "each grid row carries the decision columns")
    bop, why = best_operating_point(grid)
    ok(bop is None or (bop["ltt_valid"] and not bop["trivial"]),
       "best operating point is certified and non-trivial (or None)")
    ok(isinstance(why, str) and len(why) > 0, "best point gives a reason")
    # coverage monotone in alpha within a fixed score
    for name in sc_multi:
        rows = [r for r in grid if r["score"] == name]
        rows.sort(key=lambda r: -r["alpha"])
        covs = [r["coverage"] for r in rows]
        ok(all(covs[i] >= covs[i+1] - 1e-9 for i in range(len(covs)-1)),
           f"coverage non-increasing as alpha tightens for score {name}")

    print("[rejection breakdown]")
    b = rejection_breakdown(success, score, tau)
    ok(b["n_executed"] + b["n_rejected"] == b["n_total"], "counts partition")
    ok(b["rejected_would_fail"] + b["rejected_would_succeed"] == b["n_rejected"],
       "rejected pile splits into fail/succeed")
    ok(abs(b["over_rejection_rate"] + b["precision_of_rejection"] - 1.0) < 1e-9,
       "over-rejection and rejection-precision sum to 1")
    ok(0 <= b["over_rejection_rate"] <= 1, "over-rejection rate in [0,1]")
    line = rejection_summary_line(success, score, tau)
    ok(isinstance(line, str) and "Rejected" in line, "summary line renders")
    # a very strict threshold over-rejects; a very loose one does not
    strict = rejection_breakdown(success, score, np.quantile(score, 0.05))
    loose = rejection_breakdown(success, score, np.quantile(score, 0.95))
    ok(strict["over_rejection_rate"] >= loose["over_rejection_rate"] - 1e-9,
       "stricter threshold over-rejects at least as much")

    print("[cross-dataset comparison]")
    s_weak = success
    sc_weak = score
    s_strong = np.concatenate([np.ones(360, int), np.zeros(40, int)])[:n]
    sc_strong = rng.random(n)
    cross = cross_dataset_comparison(
        {"weak": (s_weak, sc_weak), "strong": (s_strong, sc_strong)},
        [0.15, 0.10, 0.05])
    ok(len(cross) == 2 * 3, "one row per (dataset, alpha)")
    ok(all("over_rejection" in r and "coverage" in r for r in cross),
       "cross rows carry over-rejection and coverage")
    ok(all(0 <= r["over_rejection"] <= 1 for r in cross),
       "over-rejection in [0,1] for every row")

    print("\nALL OBJECT-EVAL TESTS PASSED")


if __name__ == "__main__":
    _selftest()


In [ ]:
%%writefile configs_cornell_ggcnn.yaml
# configs/cornell_ggcnn.yaml — second detector for the model-agnostic study
model:
  name: ggcnn        # selects GG-CNN via make_model

# configs/cornell.yaml — FISAT selective-grasp, Cornell
data:
  root: /content/data/cornell     # folder holding 01..10 (+ backgrounds/)
  name: cornell
  out_size: 224
  use_depth: true                 # set false if your depth is .txt point clouds
  num_workers: 2
  split_mode: object              # folder-level; 'image' for cal/eval stage
  val_frac: 0.10
  test_frac: 0.30                 # Earlier this was 0.40 with val 0.15, which
                                  # left only 285 training images (4 folders) and
                                  # a detector too weak to succeed on ANY grasp.
                                  # Buying calibration samples is worthless if
                                  # the model being calibrated does not work.
                                  # 0.30/0.10 leaves ~530 for training.
  novel_frac: 0.0

train:
  device: cuda
  epochs: 80                      # cheap on Cornell; early stopping ends it
  batch_size: 16                  # T4-safe
  lr: 0.001
  patience: 12
  dropout: 0.1                    # >0 REQUIRED for the S3 MC-dropout score
  seed: 0

uq:
  mc_samples: 20
  methods: [deterministic, evidential]

selective:
  # Budgets chosen from the calibration-size scaling measurement (see notebook
  # section 12b), NOT from habit. At Cornell's calibration size (~150 samples)
  # alpha=0.05 certifies almost nothing and alpha=0.10 only ~20-35% coverage,
  # while alpha=0.15 reaches ~45-75%. Reporting a budget nothing can meet
  # tells the reader nothing.
  alpha: 0.15                     # primary
  alpha_secondary: 0.10           # secondary
  alpha_stretch: 0.05             # report as "not certifiable at this n"
  delta: 0.10
  n_min: 50
  splits: 100
  target_coverage: 0.80           # for the learned gate
  # gate-train / calibration / evaluation. Calibration gets the LARGEST share:
  # the gate is a small head that trains fine on less, while every certified
  # point of coverage is bought with calibration samples.
  three_way_fracs: [0.25, 0.45, 0.30]


In [ ]:
%%writefile configs_cornell_unet.yaml
# configs/cornell_unet.yaml — third detector (U-Net, skip connections)
model:
  name: unet

# configs/cornell.yaml — FISAT selective-grasp, Cornell
data:
  root: /content/data/cornell     # folder holding 01..10 (+ backgrounds/)
  name: cornell
  out_size: 224
  use_depth: true                 # set false if your depth is .txt point clouds
  num_workers: 2
  split_mode: object              # folder-level; 'image' for cal/eval stage
  val_frac: 0.10
  test_frac: 0.30                 # Earlier this was 0.40 with val 0.15, which
                                  # left only 285 training images (4 folders) and
                                  # a detector too weak to succeed on ANY grasp.
                                  # Buying calibration samples is worthless if
                                  # the model being calibrated does not work.
                                  # 0.30/0.10 leaves ~530 for training.
  novel_frac: 0.0

train:
  device: cuda
  epochs: 80                      # cheap on Cornell; early stopping ends it
  batch_size: 16                  # T4-safe
  lr: 0.001
  patience: 12
  dropout: 0.1                    # >0 REQUIRED for the S3 MC-dropout score
  seed: 0

uq:
  mc_samples: 20
  methods: [deterministic, evidential]

selective:
  # Budgets chosen from the calibration-size scaling measurement (see notebook
  # section 12b), NOT from habit. At Cornell's calibration size (~150 samples)
  # alpha=0.05 certifies almost nothing and alpha=0.10 only ~20-35% coverage,
  # while alpha=0.15 reaches ~45-75%. Reporting a budget nothing can meet
  # tells the reader nothing.
  alpha: 0.15                     # primary
  alpha_secondary: 0.10           # secondary
  alpha_stretch: 0.05             # report as "not certifiable at this n"
  delta: 0.10
  n_min: 50
  splits: 100
  target_coverage: 0.80           # for the learned gate
  # gate-train / calibration / evaluation. Calibration gets the LARGEST share:
  # the gate is a small head that trains fine on less, while every certified
  # point of coverage is bought with calibration samples.
  three_way_fracs: [0.25, 0.45, 0.30]


In [ ]:
%%writefile configs_jacquard_ggcnn.yaml
# configs/jacquard_ggcnn.yaml — ggcnn detector on Jacquard
model:
  name: ggcnn

# configs/jacquard.yaml — FISAT selective-grasp, Jacquard V2
#
# Jacquard is ~51k images / 11k objects. That scale changes the whole
# calibration story: where Cornell gives ~60 calibration samples (alpha=0.05
# uncertifiable), Jacquard gives thousands, so the strict risk budgets the paper
# wants become reachable. The cost is training time: full Jacquard is DAYS on a
# single T4. Use the max_* knobs to start small.

data:
  root: /content/data/jacquard    # folder holding JacquardV2_Dataset_0.. shards
  name: jacquard
  out_size: 224
  use_depth: true
  depth_kind: perfect             # 'perfect' (simulated) or 'stereo'
  num_workers: 2
  split_mode: object              # object hash is in the path -> EXACT disjoint
  val_frac: 0.10
  test_frac: 0.10                 # 10% of 51k is still ~5k test images
  novel_frac: 0.0
  # Development knobs. Set both to null for the full dataset. Recommended first
  # pass: max_objects ~ 800, which keeps object-disjoint splits meaningful while
  # finishing in hours, not days.
  max_objects: 800
  max_images: null

train:
  device: cuda
  epochs: 40                      # Jacquard needs fewer epochs than Cornell:
                                  # far more data per epoch, no augmentation
                                  # required (the dataset is already huge)
  batch_size: 16
  lr: 0.001
  patience: 8
  dropout: 0.1                    # >0 REQUIRED for the S3 MC-dropout score
  seed: 0

uq:
  mc_samples: 20
  methods: [deterministic, evidential]

selective:
  # With thousands of calibration samples, the stricter budgets are reachable.
  # Promote alpha=0.05 from "stretch" (as it was on Cornell) to a real target.
  alpha: 0.10                     # primary
  alpha_secondary: 0.05           # now genuinely certifiable at this n
  alpha_stretch: 0.02             # the new frontier to report
  delta: 0.10
  n_min: 50
  splits: 100
  target_coverage: 0.85
  three_way_fracs: [0.25, 0.45, 0.30]


In [ ]:
%%writefile configs_jacquard_unet.yaml
# configs/jacquard_unet.yaml — unet detector on Jacquard
model:
  name: unet

# configs/jacquard.yaml — FISAT selective-grasp, Jacquard V2
#
# Jacquard is ~51k images / 11k objects. That scale changes the whole
# calibration story: where Cornell gives ~60 calibration samples (alpha=0.05
# uncertifiable), Jacquard gives thousands, so the strict risk budgets the paper
# wants become reachable. The cost is training time: full Jacquard is DAYS on a
# single T4. Use the max_* knobs to start small.

data:
  root: /content/data/jacquard    # folder holding JacquardV2_Dataset_0.. shards
  name: jacquard
  out_size: 224
  use_depth: true
  depth_kind: perfect             # 'perfect' (simulated) or 'stereo'
  num_workers: 2
  split_mode: object              # object hash is in the path -> EXACT disjoint
  val_frac: 0.10
  test_frac: 0.10                 # 10% of 51k is still ~5k test images
  novel_frac: 0.0
  # Development knobs. Set both to null for the full dataset. Recommended first
  # pass: max_objects ~ 800, which keeps object-disjoint splits meaningful while
  # finishing in hours, not days.
  max_objects: 800
  max_images: null

train:
  device: cuda
  epochs: 40                      # Jacquard needs fewer epochs than Cornell:
                                  # far more data per epoch, no augmentation
                                  # required (the dataset is already huge)
  batch_size: 16
  lr: 0.001
  patience: 8
  dropout: 0.1                    # >0 REQUIRED for the S3 MC-dropout score
  seed: 0

uq:
  mc_samples: 20
  methods: [deterministic, evidential]

selective:
  # With thousands of calibration samples, the stricter budgets are reachable.
  # Promote alpha=0.05 from "stretch" (as it was on Cornell) to a real target.
  alpha: 0.10                     # primary
  alpha_secondary: 0.05           # now genuinely certifiable at this n
  alpha_stretch: 0.02             # the new frontier to report
  delta: 0.10
  n_min: 50
  splits: 100
  target_coverage: 0.85
  three_way_fracs: [0.25, 0.45, 0.30]


In [ ]:
import glob
# point every detector config's data root at the local DATA dir
for cf in glob.glob(f'configs_{DATASET}_*.yaml'):
    txt = open(cf).read().replace('/content/data/cornell', DATA
                        ).replace('/content/data/jacquard', DATA)
    open(cf, 'w').write(txt)
import yaml
cfg = yaml.safe_load(open(f'configs_{DATASET}_ggcnn.yaml'))
print('dataset', cfg['data']['name'], '| example detector', cfg['model']['name'])

### Offline tests, including GG-CNN's shape arithmetic

GG-CNN's transposed-conv output_padding is tuned so 224 -> 224 exactly; the
shape trace catches the pixel-shift bug before any GPU time (the same class of
bug that bit GR-ConvNet).

In [ ]:
!python models_ggcnn.py
!python metrics_grasp.py
!python train.py --smoke

## 3. Data

The detectors train on the SAME dataset as the main notebooks, so the data must
be present locally. This cell unzips it from Drive if needed and STOPS with a
clear message if it can't find it -- rather than letting training fail many
cells later with a confusing error.

In [ ]:
import glob, os

def count_images(root, dataset):
    if dataset == 'cornell':
        # Cornell RGB files are pcd*r.png; count those specifically
        return len(glob.glob(f'{root}/**/pcd*r.png', recursive=True))
    else:
        return len(glob.glob(f'{root}/**/*_RGB.png', recursive=True))

n = count_images(DATA, DATASET)
if n == 0:
    archives = sorted(f for f in glob.glob(f'{DRIVE}/*')
                      if f.lower().endswith(('.zip','.rar','.7z','.tar','.tgz','.tar.gz')))
    print('local data empty; archives on Drive:', [os.path.basename(a) for a in archives])
    for a in archives:
        low = a.lower()
        print('extracting', os.path.basename(a))
        if low.endswith('.rar'):
            # NOTE: each shell line needs its own call; '!a; !b' does NOT work.
            get_ipython().system("apt-get -qq install -y unrar >/dev/null 2>&1")
            get_ipython().system(f'unrar x -o+ -idq "{a}" "{DATA}/"')
        elif low.endswith('.zip'):
            get_ipython().system(f'unzip -q -o "{a}" -d "{DATA}"')
        elif low.endswith('.7z'):
            get_ipython().system("apt-get -qq install -y p7zip-full >/dev/null 2>&1")
            get_ipython().system(f'7z x -y -bso0 "{a}" -o"{DATA}"')
        else:
            get_ipython().system(f'tar xf "{a}" -C "{DATA}"')
    n = count_images(DATA, DATASET)

print(f'{DATASET} image files found: {n}')
if n == 0:
    raise SystemExit(
        f"No {DATASET} data under {DATA}. Upload the dataset archive to {DRIVE} "
        f"(or unpack it into {DATA}) and re-run this cell BEFORE training. "
        f"Cornell expects pcd*r.png files in 01..10 subfolders; Jacquard expects "
        f"*_RGB.png under JacquardV2_Dataset_* folders.")

## 4. Train the two extra detectors, cache their scores

A loop over GG-CNN and U-Net: each gets its own runs/ subdir and scores file, so
the GR-ConvNet artifacts from the main notebook stay untouched. Both are small
and train in minutes.

In [ ]:
import os

DETECTORS = [
    ('ggcnn', f'configs_{DATASET}_ggcnn.yaml', f'scores_{DATASET}_ggcnn.npz'),
    ('unet',  f'configs_{DATASET}_unet.yaml',  f'scores_{DATASET}_unet.npz'),
]

for name, conf, out in DETECTORS:
    rdir = f'{WORK}/runs_{name}'
    os.makedirs(rdir, exist_ok=True)
    if os.path.exists(f'{rdir}/deterministic/best.pt'):
        print(f'{name}: deterministic exists — skipping')
    else:
        get_ipython().system(
            f'python train.py --config {conf} --method deterministic '
            f'--out {rdir}')
    if os.path.exists(f'{rdir}/evidential/best.pt'):
        print(f'{name}: evidential exists — skipping')
    else:
        get_ipython().system(
            f'python train.py --config {conf} --method evidential --out {rdir}')
    # guard: training must have produced a checkpoint before we extract
    if not os.path.exists(f'{rdir}/deterministic/best.pt'):
        print(f'*** {name}: training did not produce a checkpoint. See the '
              f'training output above (usually missing data or an OOM). '
              f'Skipping {name}.')
        continue
    if os.path.exists(f'{WORK}/{out}'):
        print(f'{name}: {out} exists — skipping')
    else:
        get_ipython().system(
            f'python extract_scores.py --config {conf} '
            f'--ckpt {rdir}/deterministic/best.pt '
            f'--ev-ckpt {rdir}/evidential/best.pt '
            f'--split test --out {out}')
    if not os.path.exists(f'{WORK}/{out}'):
        print(f'*** {name}: score extraction failed (no {out}). '
              f'See extract output above. Skipping {name}.')
        continue
    get_ipython().system(f'cp {out} "{DRIVE}/{out}" 2>/dev/null')
    import numpy as np
    d = np.load(f'{WORK}/{out}')
    _sr = float(d["success"].mean())
    print(name + ": N=" + str(len(d["success"])) + " success=" + format(_sr, ".3f"))
    print()

## 5. The model-agnostic comparison

Run the SAME rejection analysis on both detectors' caches and compare. The claim
holds if the LTT guarantee is satisfied on GG-CNN too, and the risk-coverage
behaviour is qualitatively the same (the method doesn't depend on the
architecture).

In [ ]:
import selective_grasp as sg, importlib
importlib.reload(sg)
from selective_grasp import (load_scores, aurc, coverage_at_risk,
                             ltt_calibrate_single, to_markdown)

ALPHA, DELTA = cfg['selective']['alpha'], cfg['selective']['delta']

def summarize(path, label):
    scores, loss = load_scores(path)
    best = min(scores, key=lambda k: aurc(scores[k], loss))
    res = ltt_calibrate_single(scores[best], loss, ALPHA, DELTA)
    cov = coverage_at_risk(scores[best], loss, ALPHA)
    return dict(detector=label, N=len(loss),
                success=round(1-float(loss.mean()),3),
                best_score=best, best_AURC=round(aurc(scores[best],loss),4),
                LTT_valid=bool(res.valid),
                coverage_at_alpha=round(cov,3))

import os
rows = []
for name, path in [('GG-CNN', f'scores_{DATASET}_ggcnn.npz'),
                   ('U-Net',  f'scores_{DATASET}_unet.npz')]:
    if os.path.exists(path):
        rows.append(summarize(path, name))
# GR-ConvNet reference cache from the main notebook for THIS dataset
gr = f'{DRIVE}/scores.npz'
if os.path.exists(gr):
    rows.insert(0, summarize(gr, 'GR-ConvNet'))
else:
    print(f'GR-ConvNet scores.npz not found at {gr} — showing new ones only.')
print(f"=== {DATASET.upper()} ===")
print(to_markdown(rows))
print()
valid = [r for r in rows if r['LTT_valid']]
print(f"LTT certified risk <= alpha on {len(valid)}/{len(rows)} detectors.")
if len(valid) == len(rows) and len(rows) >= 3:
    print("-> the guarantee holds across all three architectures: the rejection")
    print("   rule is demonstrably model-agnostic.")

### Reading it

- If **LTT_valid** is True for both detectors, the risk guarantee is
  architecture-independent -- the core claim.
- The two detectors will differ in raw accuracy and in which score wins; that is
  expected and fine. What matters is that the SAME procedure certifies a budget
  on both.
- A short line for the paper: "Applied unchanged to GG-CNN, a detector with ~Nx
  fewer parameters, the rejection rule certified risk <= alpha with coverage
  comparable to GR-ConvNet, demonstrating the method is agnostic to the
  underlying detector."

## 6. Save

In [ ]:
import os
os.makedirs(f'{DRIVE}/ggcnn', exist_ok=True)
!rsync -a "{WORK}/runs/" "{DRIVE}/ggcnn/runs/" 2>/dev/null
!cp "{WORK}/scores_ggcnn.npz" "{DRIVE}/scores_ggcnn.npz" 2>/dev/null
print('saved GG-CNN artifacts to Drive')